# MC vs Split Conformal Prediction 시뮬레이션

CLAUDE.md / 시뮬.md 스펙 기준 구현. 한 셀씩 같이 논의하면서 진행.

**현재까지 들어간 것**: RNG 독립 스트림 → 오차분포 3종 → DGP(mean/scale) → 모델 학습(OLS/RF)


In [1]:
import math
import numpy as np
import pandas as pd
from scipy import stats, integrate, optimize
from dataclasses import dataclass
from typing import Callable

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

## 1. RNG (난수 생성기) 독립 스트림

**rng란?** 난수를 차례대로 뽑아주는 객체. 같은 seed로 만들면 언제 실행해도 같은 순서의 값이 나오므로(재현 가능), 실험을 매번 또같이 돌려볼 수 있음.

**왜 하나의 rng를 계속 재사용하면 안 되는가**: 예를 들어 training 데이터와 calibration 데이터를 같은 rng로 순차적으로 만들면, training 쪽 코드를 조금만 고쳐도(난수를 하나라도 더 뽑거나 덜 뽑거나) calibration 데이터까지 달라져버림.

**해결책**: `substream(scenario, b, stage)`처럼 이름표(path)를 붙여서, 이름표가 다르면 서로 완전히 독립적인 rng를 만드는 함수를 만듦. 같은 이름표 → 같은 결과(재현), 다른 이름표 → 서로 무관한 다른 결과(독립).

문자열을 숫자로 바꿀 때는 파이썬 내장 `hash()`를 쓰지 않음(실행마다 값이 바뀌는 경우가 있어 재현성이 깨짐) — 대신 `hashlib.sha256`으로 실행 환경과 무관하게 액상을 고정된 수로 바꿈 (CLAUDE.md §9).

In [2]:
import hashlib

# 이 값이 같으면 전체 실험이 항상 또같이 재현됨
MASTER_SEED = 20240911


def stable_int(value):
    """문자열/정수를 항상 같은 정수로 바꿀 (파이썬 hash()는 실행마다 바뀌어서 사용 금지)."""
    text = str(value)
    digest = hashlib.sha256(text.encode("utf-8")).digest()  # 문자열 -> 32바이트 고정 지문값
    return int.from_bytes(digest[:8], "little")  # 그 중 8바이트만 정수로 변환


def substream(*path):
    """(scenario, b, r, stage, ...) 같은 이름표로 독립적인 rng를 생성.

    같은 path -> 항상 같은 난수열 (재현 가능)
    다른 path -> 서로 독립적인 다른 난수열
    """
    entropy = [MASTER_SEED] + [stable_int(p) for p in path]
    seed_sequence = np.random.SeedSequence(entropy)
    return np.random.default_rng(seed_sequence)

In [3]:
# 간단한 확인: 같은 path는 같은 값, 다른 path(시나리오/stage)는 다른 값
rng_a = substream("linear_homo_gaussian", 1, "train")
rng_b = substream("linear_homo_gaussian", 1, "train")
rng_c = substream("linear_homo_gaussian", 1, "model")  # stage만 다름

print("rng_a :", rng_a.standard_normal(3))
print("rng_b :", rng_b.standard_normal(3))
print("rng_c :", rng_c.standard_normal(3))

assert np.array_equal(rng_a.standard_normal(0), rng_b.standard_normal(0))  # 둘 다 빈 배열, 형태만 확인
print("RNG 기본 성질 확인 완료")


rng_a : [-0.578851  0.592685  1.246618]
rng_b : [-0.578851  0.592685  1.246618]
rng_c : [0.064885 2.014959 1.352782]
RNG 기본 성질 확인 완료


## 2. Error distribution

각 error는 `sample(size, rng)`, `cdf(e)`, `ppf(p)` 세 메서드를 갖는 클래스. 이 세 메서드가 같은 표준화 상수(df, scale, a, s 등)를 공유해야 해서 함수 대신 클래스로 묶음.

- `sample(size, rng)`: 실제 난수 표본을 size개 뽑음
- `cdf(e)`: $P(\varepsilon \le e)$. coverage 계산에 쓰임
- `ppf(p)`: cdf의 역함수(분위수). 예를 들어 `ppf(0.975)`는 상위 2.5% 지점 값

세 분포 모두 $E[\varepsilon]=0$, $\mathrm{Var}(\varepsilon)=1$로 표준화됨:
- Gaussian: 정규분포
- Student-$t$: 대칭이지만 꼬리가 두꺼운 분포
- Lognormal: 비대칭 분포

In [4]:
class GaussianError:
    """E[eps]=0, Var(eps)=1인 표준정규 오차. 표준정규는 원래부터 평균0·분산1이라 별도 변환이 필요 없음."""

    def sample(self, size, rng):
        # 실제 난수를 size개 생성 (몬테카를로 표본, training/calibration의 잡음 등에 사용)
        return rng.standard_normal(size)

    def cdf(self, e):
        # cumulative distribution function: P(eps <= e). coverage 계산에 사용
        return stats.norm.cdf(e)

    def ppf(self, p):
        # cdf의 역함수(quantile function): P(eps <= q) = p 인 q를 반환
        # 예: ppf(0.975) -> 상위 2.5% 지점 값. Oracle 구간의 upper bound 계산에 사용
        return stats.norm.ppf(p)


class StudentTError:
    """eps = T / sqrt(3), T ~ t_3. t_3의 분산이 3이므로 sqrt(3)로 나누어 분산 1로 표준화."""

    def __init__(self, df=3):
        self.df = df  # 자유도(degrees of freedom); 작을수록 꼬리가 두꺼움(heavy tail)
        # t_df 분포의 분산은 df/(df-2). 이 값으로 나눠주면 최종 eps의 분산이 정확히 1이 됨
        self.scale = np.sqrt(df / (df - 2))

    def sample(self, size, rng):
        raw_t_sample = rng.standard_t(self.df, size=size)  # 아직 분산이 1이 아닌 원래 t분포 표본
        return raw_t_sample / self.scale  # scale로 나눠서 분산을 1로 맞춤

    def cdf(self, e):
        # 우리 eps는 원래 t분포를 scale로 나눈 것이므로,
        # 반대로 e에 scale을 다시 곱하면 원래 t분포 값으로 되돌아감 -> 거기서 t분포의 cdf를 사용
        e_in_original_t_scale = np.asarray(e) * self.scale
        return stats.t.cdf(e_in_original_t_scale, df=self.df)

    def ppf(self, p):
        q_in_original_t_scale = stats.t.ppf(p, df=self.df)  # 원래 t분포 기준의 분위수
        return q_in_original_t_scale / self.scale  # 우리 eps 기준(분산 1)으로 다시 변환


class LogNormalError:
    """eps = (exp(Z) - a) / s, Z~N(0,1), a=exp(1/2), s=sqrt(e*(e-1)).

    exp(Z) 자체는 평균이 0도 아니고 분산도 1이 아니므로(로그정규분포),
    평균을 a만큼 빼고 표준편차 s로 나눠서 최종 eps의 평균 0·분산 1을 맞춤.
    """

    def __init__(self):
        self.a = np.exp(0.5)  # exp(Z)의 평균 (Z~N(0,1)일 때 E[exp(Z)] = exp(1/2))
        self.s = np.sqrt(np.e * (np.e - 1))  # exp(Z)의 표준편차

    def sample(self, size, rng):
        z = rng.standard_normal(size)  # 먼저 표준정규 Z를 뽑고
        raw_lognormal = np.exp(z)  # Z를 exp에 넣으면 로그정규분포가 됨 (평균 0·분산 1 아님)
        return (raw_lognormal - self.a) / self.s  # 평균 0·분산 1로 표준화

    def cdf(self, u):
        # 목표: eps <= u 일 확률을 구하는 것
        # 1단계: 우리 eps 기준값 u를, 표준화하기 전의 "원래 exp(Z)" 기준값으로 되돌림
        #        eps = (exp(Z)-a)/s 였으니 반대로 풀면 exp(Z) = s*u + a
        u = np.asarray(u, dtype=float)
        original_scale_value = self.s * u + self.a  # 이게 바로 "s*u+a" = 원래 exp(Z) 기준값

        # 2단계: exp(Z) <= original_scale_value 일 확률을 구하면 됨.
        #        exp(Z)는 항상 양수이므로, original_scale_value <= 0이면 그 확률은 무조건 0
        is_in_support = original_scale_value > 0

        # 3단계: original_scale_value가 0보다 클 때만 log를 취해도 안전함.
        #        0 이하인 곳은 나중에 버릴 값이지만, log(0 이하)는 계산 자체가 에러/경고를 내므로
        #        일단 안전한 더미값 1.0으로 채워서 계산만 통과시킴 (결과는 다음 줄에서 덮어씀)
        safe_value_for_log = np.where(is_in_support, original_scale_value, 1.0)

        # 4단계: exp(Z) <= val  <=>  Z <= log(val) 이므로, 표준정규 cdf(log(val))가 원하는 확률
        probability_if_in_support = stats.norm.cdf(np.log(safe_value_for_log))

        # 5단계: support 밖(원래 값<=0)이었던 자리는 확률 0으로 덮어씀
        result = np.where(is_in_support, probability_if_in_support, 0.0)

        # 6단계: 입력 자체가 NaN이었던 자리는 결과도 NaN으로 유지 (0으로 몰래 바꾸지 않음)
        return np.where(np.isnan(u), np.nan, result)

    def ppf(self, p):
        # cdf의 역순으로 계산: 표준정규 분위수 -> exp로 원래 스케일 -> 다시 (a,s)로 표준화
        p = np.asarray(p, dtype=float)
        standard_normal_quantile = stats.norm.ppf(p)  # Phi^{-1}(p)
        original_scale_value = np.exp(standard_normal_quantile)  # exp(Z) 기준의 분위수
        return (original_scale_value - self.a) / self.s  # 표준화된 eps 기준으로 변환


ERROR_DISTRIBUTIONS = {
    "gaussian": GaussianError(),
    "student_t": StudentTError(),
    "lognormal": LogNormalError(),
}

### 검증: mean≈0, var≈1, cdf(ppf(p))≈p, LogNormal support 경계

In [5]:
rng_check = substream("validation", "error_distribution")
n_check = 2_000_000
p_grid = np.linspace(0.01, 0.99, 25)

for name, err in ERROR_DISTRIBUTIONS.items():
    samples = err.sample(n_check, substream("validation", "error_distribution", name))
    cdf_ppf_err = np.max(np.abs(err.cdf(err.ppf(p_grid)) - p_grid))
    print(f"{name:10s} mean={samples.mean():+.4f}  var={samples.var():.4f}  max|cdf(ppf(p))-p|={cdf_ppf_err:.2e}")

ln = ERROR_DISTRIBUTIONS["lognormal"]
boundary = -ln.a / ln.s
print("\nlognormal support 경계:", "미만", ln.cdf(boundary - 0.01), " / ", "이상", ln.cdf(boundary + 0.01))


gaussian   mean=-0.0003  var=1.0000  max|cdf(ppf(p))-p|=1.11e-16
student_t  mean=+0.0008  var=0.9878  max|cdf(ppf(p))-p|=2.22e-16
lognormal  mean=-0.0011  var=0.9914  max|cdf(ppf(p))-p|=1.11e-16

lognormal support 경계: 미만 0.0  /  이상 6.290800057141992e-05


## 3. DGP (Data Generating Process)

$X=(X_1,X_2)$, $X_1,X_2\overset{iid}{\sim}U(-1,1)$, $Y=m_0(X)+\sigma_0(X)\varepsilon$

- `m0_linear`, `m0_nonlinear`: true conditional mean (평균 0, 분산 4/3으로 통일)
- `sigma0_homo`, `sigma0_hetero`: true conditional scale. hetero는 $X_1$에만 의존
- `draw_X`, `draw_Y`: 설명변수와 반응변수 생성

In [6]:
def m0_linear(X):
    x1, x2 = X[:, 0], X[:, 1]
    return np.sqrt(2) * (x1 + x2)


def m0_nonlinear(X):
    # X1은 비선형(sin), X2는 선형, x1*x2 상호작용 포함.
    # OLS는 절편+X1+X2만 쓰므로 이 경우 OLS에 misspecification(모형 오설정)이 생김
    x1, x2 = X[:, 0], X[:, 1]
    return np.sqrt(6 / 11) * (2 * np.sin(np.pi * x1) + x2 + x1 * x2)


def sigma0_homo(X):
    return np.ones(X.shape[0])  # 모든 위치에서 noise 크기 동일(등분산)


def sigma0_hetero(X):
    x1 = X[:, 0]
    return 0.4 + 1.2 * np.sin(np.pi * x1) ** 2  # X1에 따라 noise 크기가 달라짐(이분산)


MEAN_FUNCTIONS = {"linear": m0_linear, "nonlinear": m0_nonlinear}
SCALE_FUNCTIONS = {"homo": sigma0_homo, "hetero": sigma0_hetero}


def draw_X(n, rng):
    return rng.uniform(-1.0, 1.0, size=(n, 2))  # (n, 2) shape, 각 열이 U(-1,1)에서 독립


def draw_Y(X, mean_fn, scale_fn, error, rng):
    eps = error.sample(X.shape[0], rng)  # X와 독립적인 잡음
    return mean_fn(X) + scale_fn(X) * eps


### 검증: mean 평균 0·분산 4/3, scale 최소 양수·평균제곱 1(homo)/1.18(hetero)

In [7]:
rng_dgp_check = substream("validation", "dgp")
n_check = 5_000_000
X_check = draw_X(n_check, rng_dgp_check)

for name, fn in MEAN_FUNCTIONS.items():
    m = fn(X_check)
    print(f"m0_{name:10s} mean={m.mean():+.5f} (target 0)   var={m.var():.5f} (target {4/3:.4f})")

for name, fn in SCALE_FUNCTIONS.items():
    s = fn(X_check)
    target = 1.0 if name == "homo" else 1.18
    print(f"sigma0_{name:8s} min={s.min():.5f} (>0)   mean(sigma^2)={np.mean(s**2):.5f} (target {target})")


m0_linear     mean=+0.00049 (target 0)   var=1.33376 (target 1.3333)
m0_nonlinear  mean=+0.00068 (target 0)   var=1.33376 (target 1.3333)
sigma0_homo     min=1.00000 (>0)   mean(sigma^2)=1.00000 (target 1.0)
sigma0_hetero   min=0.40000 (>0)   mean(sigma^2)=1.18016 (target 1.18)


### DGP 컨테이너

`mean_fn`, `scale_fn`, `error`를 하나의 시나리오로 묶음. 2 mean × 2 scale × 3 error = 12개 DGP.

In [8]:
@dataclass
class DGP:
    name: str
    mean_fn: Callable
    scale_fn: Callable
    error: object
    error_name: str  # rng_mc를 error별로 공유할 때 쓸 이름 (DGP와 불일치 방지용)
    scale_name: str  # q0(res_true 반폭)가 mean과 무관하게 scale·error에만 의존해서 캐시할 때 쓸 이름

    def sample_xy(self, n, rng):
        X = draw_X(n, rng)
        Y = draw_Y(X, self.mean_fn, self.scale_fn, self.error, rng)
        return X, Y


DGPS = {}
for mean_name, mean_fn in MEAN_FUNCTIONS.items():
    for scale_name, scale_fn in SCALE_FUNCTIONS.items():
        for error_name, error in ERROR_DISTRIBUTIONS.items():
            dgp_name = f"{mean_name}_{scale_name}_{error_name}"
            DGPS[dgp_name] = DGP(dgp_name, mean_fn, scale_fn, error, error_name, scale_name)

print(f"총 {len(DGPS)}개 DGP")
DGPS

총 12개 DGP


{'linear_homo_gaussian': DGP(name='linear_homo_gaussian', mean_fn=<function m0_linear at 0x000002AD3CDBDE40>, scale_fn=<function sigma0_homo at 0x000002AD6C40A200>, error=<__main__.GaussianError object at 0x000002AD7F5AF620>, error_name='gaussian', scale_name='homo'),
 'linear_homo_student_t': DGP(name='linear_homo_student_t', mean_fn=<function m0_linear at 0x000002AD3CDBDE40>, scale_fn=<function sigma0_homo at 0x000002AD6C40A200>, error=<__main__.StudentTError object at 0x000002AD7F5AF4D0>, error_name='student_t', scale_name='homo'),
 'linear_homo_lognormal': DGP(name='linear_homo_lognormal', mean_fn=<function m0_linear at 0x000002AD3CDBDE40>, scale_fn=<function sigma0_homo at 0x000002AD6C40A200>, error=<__main__.LogNormalError object at 0x000002AD7F5AF770>, error_name='lognormal', scale_name='homo'),
 'linear_hetero_gaussian': DGP(name='linear_hetero_gaussian', mean_fn=<function m0_linear at 0x000002AD3CDBDE40>, scale_fn=<function sigma0_hetero at 0x000002AD6C40AB60>, error=<__main__

## 4. 모델 학습 (OLS, RF)

- OLS: 절편 + $X_1,X_2$만 사용
- RF: squared-error 회귀 forest, 1000 trees. 나머지 하이퍼파라미터는 sklearn 기본값을 그대로 쓰을 pilot 기본안으로 명시

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

RF_HYPERPARAMS = dict(
    n_estimators=1000,
    criterion="squared_error",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    bootstrap=True,
    n_jobs=1,  # 밖에서 b(training 반복)를 병렬화할 예정이라 RF 내부 병렬화는 꺼서 oversubscription 방지
)


def fit_ols(X, y):
    return LinearRegression().fit(X, y)


def fit_rf(X, y, rng):
    seed = int(rng.integers(0, 2**31 - 1))  # rng에서 정수 하나를 뽑아 sklearn의 random_state로 사용
    return RandomForestRegressor(random_state=seed, **RF_HYPERPARAMS).fit(X, y)


### 확인: linear_homo_gaussian에서 학습해보기

$n_{train}=1000$으로 training 데이터를 만들고 OLS·RF를 학습. 데이터 생성용 rng와 RF 학습용 rng를 다른 stage로 분리해서 독립적으로 쓰는다.

In [10]:
dgp = DGPS["linear_homo_gaussian"]
b = 1

rng_train = substream(dgp.name, b, "train")
rng_model = substream(dgp.name, b, "model")

X_train, y_train = dgp.sample_xy(1000, rng_train)
ols_model = fit_ols(X_train, y_train)
rf_model = fit_rf(X_train, y_train, rng_model)

print("OLS intercept:", ols_model.intercept_, "(target 0)")
print("OLS coef     :", ols_model.coef_, f"(target both {np.sqrt(2):.4f})")

x_eval = np.array([[0.0, 0.0], [0.5, -0.5], [0.9, 0.9]])
print("\ntrue m0    :", dgp.mean_fn(x_eval))
print("OLS predict:", ols_model.predict(x_eval))
print("RF  predict:", rf_model.predict(x_eval))

OLS intercept: -0.015698804450583234 (target 0)
OLS coef     : [1.345043 1.456035] (target both 1.4142)

true m0    : [0.       0.       2.545584]
OLS predict: [-0.015699 -0.071195  2.505271]
RF  predict: [-0.309961 -0.518883  2.387691]


## 5. Oracle, MC 구간

- `oracle_interval`: true 오차분포의 분위수로 정확히 계산한 구간. $[m_0(x)+\sigma_0(x)q_{\alpha/2},\ m_0(x)+\sigma_0(x)q_{1-\alpha/2}]$
- `mc_interval`: 실제로 오차 표본을 $M_{MC}$개 뽑아서, 그 표본의 empirical quantile로 만든 구간 (oracle을 몬테카를로로 근사)
- `conditional_coverage`: 구간 $[L,U]$가 특정 위치 $x$에서 새 $Y$를 포함할 확률. $\{L < Y<U \}=\{L<m_0(x)+\sigma_0(x)\epsilon<U\}=\{\frac{L-m_0(x)}{\sigma_0(x)}<\epsilon<\frac{U-m_0(x)}{\sigma_0(x)}\}\Rightarrow F_\varepsilon\!\left(\frac{U-m_0(x)}{\sigma_0(x)}\right)-F_\varepsilon\!\left(\frac{L-m_0(x)}{\sigma_0(x)}\right)$ — CDF로 직접 계산하므로 새 $Y$를 뽑을 필요 없음 (CLAUDE.md §6.1, §9)

MC 표본(`eps_sample`)은 error 분포에서만 뽑고, mean/scale은 그 뒤에 위치·크기 변환으로 적용 — 그래서 같은 표본을 모든 $x$에 재사용 가능 (시뮬.md §4.3).

In [11]:
def oracle_interval(X, dgp, alpha):
    """true 분위수를 그대로 사용해서 이론적으로 정확한 구간을 만든다.

    구간 = [m0(x) + sigma0(x)*q_lo, m0(x) + sigma0(x)*q_hi]
    여기서 q_lo, q_hi는 진짜 오차분포 eps의 양쪽 분위수.
    """
    # 1단계: 이 x에서 true mean, true scale을 계산 (우리가 DGP를 직접 설계했으니 정확히 알고 있음)
    true_mean = dgp.mean_fn(X)
    true_scale = dgp.scale_fn(X)

    # 2단계: 목표 alpha에 맞는 양쪽 분위수를 구함
    #        alpha=0.05면 q_lo=하위 2.5% 지점, q_hi=상위 97.5% 지점(=하위 2.5%를 뺀 지점)
    q_lo = dgp.error.ppf(alpha / 2)
    q_hi = dgp.error.ppf(1 - alpha / 2)

    # 3단계: 위치(true_mean)에 폭(true_scale * 분위수)을 더해서 구간의 양 끝을 만듦
    lower_bound = true_mean + true_scale * q_lo
    upper_bound = true_mean + true_scale * q_hi
    return lower_bound, upper_bound


def draw_mc_benchmark(dgp, M_MC, rng):
    """MC 구간을 만드는 데 쓸 오차 표본을 M_MC개 뽑는다.

    이 표본은 error 분포에서만 뽑히고 mean/scale과는 무관하므로,
    같은 표본을 여러 x 위치에 재사용할 수 있다 (mc_interval에서 위치·크기 변환만 적용).
    """
    return dgp.error.sample(M_MC, rng)


def mc_interval(X, dgp, alpha, eps_sample):
    """오차 표본(eps_sample)의 empirical quantile로 oracle 구간을 몬테카를로 근사한다."""
    # 1단계: 표본에서 empirical quantile(경험적 분위수)을 계산.
    #        method="linear"는 numpy의 기본 보간 방식 -> 이 보간 규칙을 고정해서 기록해둠
    q_lo, q_hi = np.quantile(eps_sample, [alpha / 2, 1 - alpha / 2], method="linear")

    # 2단계: oracle_interval과 똑같은 방식으로, true mean/scale에 이 분위수를 적용
    true_mean = dgp.mean_fn(X)
    true_scale = dgp.scale_fn(X)
    lower_bound = true_mean + true_scale * q_lo
    upper_bound = true_mean + true_scale * q_hi
    return lower_bound, upper_bound


def conditional_coverage(L, U, X, dgp):
    """구간 [L,U]가 특정 위치 x에서 실제로 새 Y를 포함할 확률.

    새로운 Y를 무작위로 많이 뽑아서 세는 방식이 아니라, true CDF로 정확히 계산한다
    (CLAUDE.md §6.1, §9: coverage는 CDF로 직접 계산하고 이진 포함 횟수 추정으로 대체하지 않음).
    """
    true_mean = dgp.mean_fn(X)
    true_scale = dgp.scale_fn(X)

    # Y = m0(X) + sigma0(X)*eps 이므로, L <= Y <= U  <=>  (L-m0)/sigma0 <= eps <= (U-m0)/sigma0
    # 즉 끝점 L, U를 "표준화된 eps 기준"으로 바꾼 뒤, 그 사이에 eps가 들어올 확률을 CDF 차이로 구함
    standardized_lower = (L - true_mean) / true_scale
    standardized_upper = (U - true_mean) / true_scale
    return dgp.error.cdf(standardized_upper) - dgp.error.cdf(standardized_lower)

### 확인: oracle vs mc, 같은 x에서 비교

In [12]:
dgp = DGPS["nonlinear_hetero_lognormal"]  # 가장 어려운 조합(이분산 + 비대칭 오차)으로 확인
alpha = 0.05
M_MC = 100_000

eps_sample = draw_mc_benchmark(dgp, M_MC, substream("mc_benchmark", dgp.error_name))

x_eval = np.array([[0.0, 0.0], [0.9, 0.9]])
L_oracle, U_oracle = oracle_interval(x_eval, dgp, alpha)
L_mc, U_mc = mc_interval(x_eval, dgp, alpha, eps_sample)

print("oracle:", np.stack([L_oracle, U_oracle], axis=1))
print("mc    :", np.stack([L_mc, U_mc], axis=1))
print("endpoint 차이 (L,U):", L_mc - L_oracle, U_mc - U_oracle)
print("oracle coverage:", conditional_coverage(L_oracle, U_oracle, x_eval, dgp), "(target 0.95)")
print("mc     coverage:", conditional_coverage(L_mc, U_mc, x_eval, dgp))

oracle: [[-0.279078  1.008765]
 [ 1.36034   3.017117]]
mc    : [[-0.27933   1.005121]
 [ 1.360016  3.01243 ]]
endpoint 차이 (L,U): [-0.000252 -0.000324] [-0.003644 -0.004688]
oracle coverage: [0.95 0.95] (target 0.95)
mc     coverage: [0.950399 0.950399]


### $M_{MC}$ 탐색: MC 분위수가 true 분위수에 얼마나 가까워지는가

MC 구간의 정확도는 mean/scale과 무관하게 **error 분포 + $M_{MC}$ + $\alpha$**로만 결정됨 (mean/scale은 나중에 곱하고 더하는 변환일 뿐이라서). 그래서 raw 분위수 오차 $|\hat q - q_{true}|$를 직접 보는 게 가장 간단.

$\alpha=0.01$(양쪽 0.5%)처럼 극단적인 분위수, 그리고 Student-$t$·LogNormal처럼 꼬리가 무겁거나 비대칭인 분포가 가장 느리게 수렴할 것으로 예상 — 이 조합으로 $M_{MC}$를 늘려가며 확인.

In [13]:
M_MC_CANDIDATES = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]
ALPHAS = [0.10, 0.05, 0.01]
N_REPS = 5  # 같은 M_MC라도 반복마다 오차가 얼마나 흔들리는지 보기 위해 여러 번 반복

rows = []  # 표로 만들 결과를 한 줄씩 여기에 쌓음

# 바깥 루프: error 분포 3종류를 하나씩 확인
for error_name, err in ERROR_DISTRIBUTIONS.items():
    # 그 다음 루프: alpha(목표 coverage 수준) 3종류를 하나씩 확인
    for alpha in ALPHAS:
        # 이 error, 이 alpha에서의 "정답" 분위수 (M_MC와 무관하게 고정된 진짜 값)
        q_lo_true = err.ppf(alpha / 2)
        q_hi_true = err.ppf(1 - alpha / 2)

        # 그 다음 루프: M_MC(표본 개수)를 늘려가며 확인
        for M_MC in M_MC_CANDIDATES:
            worst_err = 0.0  # 이 (error, alpha, M_MC) 조합에서 N_REPS번 중 가장 컸던 오차를 기록

            # 가장 안쪽 루프: 같은 조건을 N_REPS번 반복해서, 운이 좋아서 우연히 잘 맞은 건 아닌지 확인
            for rep in range(N_REPS):
                rng = substream("mc_pilot", error_name, alpha, M_MC, rep)  # 반복마다 독립적인 rng
                sample = err.sample(M_MC, rng)  # 실제로 M_MC개의 오차 표본을 뽑음
                q_lo, q_hi = np.quantile(sample, [alpha / 2, 1 - alpha / 2], method="linear")

                # 이번 반복에서의 오차(추정값과 정답의 차이)를 계산해서, 지금까지의 최댓값과 비교
                error_lo = abs(q_lo - q_lo_true)
                error_hi = abs(q_hi - q_hi_true)
                worst_err = max(worst_err, error_lo, error_hi)

            rows.append({"error": error_name, "alpha": alpha, "M_MC": M_MC, "worst_err": worst_err})

# 쌓아둔 결과를 표로 만듦: error x alpha를 행으로, M_MC를 열로 펼쳐서 한눈에 비교
mc_error_table = pd.DataFrame(rows).pivot(index=["error", "alpha"], columns="M_MC", values="worst_err")
mc_error_table

M_MC              1000      10000     100000    1000000   10000000
error     alpha                                                   
gaussian  0.0100    0.3719    0.1169    0.0365    0.0132    0.0025
          0.0500    0.1700    0.0408    0.0152    0.0032    0.0018
          0.1000    0.0798    0.0381    0.0219    0.0058    0.0011
lognormal 0.0100    1.1907    0.7288    0.0915    0.0617    0.0121
          0.0500    0.5504    0.1400    0.0369    0.0072    0.0063
          0.1000    0.3340    0.0860    0.0323    0.0084    0.0047
student_t 0.0100    1.1262    0.2405    0.1048    0.0276    0.0128
          0.0500    0.2509    0.0760    0.0239    0.0101    0.0020
          0.1000    0.2327    0.0508    0.0132    0.0051    0.0020

## 6. `res_true` 구간 ($q_{0,\alpha}$: population residual quantile)

True mean $m_0$은 그대로 쓰지만, 폭은 **전체 $X$ 분포에서 나온 residual** $R_0=|Y-m_0(X)|=|\sigma_0(X)\varepsilon|$의 $(1-\alpha)$ population quantile $q_{0,\alpha}$ 하나로 고정(CP 방식으로 score는 계산하는데, fitted model 대신 true model을 쓰고, finite calibration 대신 무한 calibration population을 쓴 것):

$$C_{res,0}(x) = [m_0(x)-q_{0,\alpha},\ m_0(x)+q_{0,\alpha}]$$

$q_{0,\alpha}$를 구하는 식: $H(q) = E_X\!\left[F_\varepsilon(q/\sigma_0(X)) - F_\varepsilon(-q/\sigma_0(X))\right] = 1-\alpha$

- $\sigma_0(X)$가 $X_1$에만 의존하므로(homo·hetero 둘 다), $H(q)$는 **$X_1$에 대한 1차원 적분**으로 정확히 계산 가능 (`scipy.integrate.quad`)
- 그 적분값이 목표($1-\alpha$)와 같아지는 $q$를 root-finding으로 찾음 (`scipy.optimize.brentq`)
- $q_{0,\alpha}$는 **mean과 무관** — scale·error·alpha가 같으면 항상 같은 값이라 캐시 가능

In [14]:
# (scale_name, error_name, alpha) -> 이미 계산해둔 q0 값. mean과는 무관하므로 DGP 이름 전체가 아니라
# 이 세 가지로만 캐시하면 됨 (같은 scale/error/alpha면 mean이 달라도 항상 같은 q0)

# q 후보 선택 → H(q) 계산 → 1-α와 비교 → q 조정
_Q0_CACHE = {}

def _probability_within_q(q, x1, scale_fn, error):
    """위치 x1 하나에서, |R0| <= q 일 확률을 계산하는 가장 작은 단위 함수.

    R0 = |Y - m0(X)| = |sigma0(X) * eps| 이므로,
    |R0| <= q  <=>  -q/sigma0(X) <= eps <= q/sigma0(X)
    이 확률을 CDF 차이로 구한다.
    """
    # scale_fn은 (n,2) 모양의 X를 받아서 X[:,0](=x1)만 사용하므로, x2 자리는 0으로 채워도 결과에 영향 없음
    X_dummy = np.array([[x1, 0.0]])
    sigma0_at_x1 = scale_fn(X_dummy)[0]

    upper_side = error.cdf(q / sigma0_at_x1)
    lower_side = error.cdf(-q / sigma0_at_x1)
    return upper_side - lower_side


def _H_res_true(q, scale_fn, error):
    """H(q) = E_X1[ |R0| <= q 일 확률 ]=int_{-1}^{1} f(X_1)P(|R_0| ≤ q)dX_1. X1을 모든 위치에서 평균낸 것.

    X1 ~ U(-1,1)이므로 pdf가 항상 1/2. scipy.integrate.quad로 -1~1 구간을 적분해서
    "모든 X1에서의 확률"을 정확하게(수치적분으로) 평균낸다.
    """
    def integrand(x1):
        probability_at_this_x1 = _probability_within_q(q, x1, scale_fn, error)
        return probability_at_this_x1 * 0.5  # U(-1,1)의 밀도 1/2을 곱해야 진짜 적분(평균)이 됨

    integral_value, _ = integrate.quad(integrand, -1, 1)
    return integral_value


def compute_q0(scale_name, error_name, alpha):
    """H(q) = 1-alpha 를 만족하는 q0를 찾는다 (res_true 구간의 반폭).

    H(q)는 q가 커질수록 커지는(단조증가) 함수이므로 -- q=0이면 확률 0, q가 아주 크면 확률 1에 가까움 --
    이분법 계열의 root-finding(brentq)으로 정확히 그 값을 찾을 수 있다.
    """
    cache_key = (scale_name, error_name, alpha)
    if cache_key in _Q0_CACHE:
        return _Q0_CACHE[cache_key]  # 이미 계산해둔 값이면 다시 계산하지 않고 재사용

    scale_fn = SCALE_FUNCTIONS[scale_name]
    error = ERROR_DISTRIBUTIONS[error_name]
    target_probability = 1 - alpha

    # H(q) - target = 0 이 되는 q를 찾는 것. brentq(함수, 하한, 상한)은
    # "하한에서는 함수값이 음수, 상한에서는 양수(또는 그 반대)"인 구간 안에서 정확한 근을 찾아준다.
    # q=1e-8(거의 0)이면 H≈0 이라서 target보다 작고, q=50이면 H≈1이라서 target보다 크므로 근이 이 사이에 있음이 보장됨.
    def h_minus_target(q):
        return _H_res_true(q, scale_fn, error) - target_probability

    q0 = optimize.brentq(h_minus_target, 1e-8, 50.0, xtol=1e-10)

    _Q0_CACHE[cache_key] = q0  # 다음에 같은 (scale, error, alpha) 조합이 오면 재사용
    return q0


def res_true_interval(X, dgp, alpha):
    """[m0(x) - q0, m0(x) + q0]. true mean은 그대로 쓰고, 폭만 전체 X에서 나온 q0로 고정."""
    q0 = compute_q0(dgp.scale_name, dgp.error_name, alpha)
    true_mean = dgp.mean_fn(X)
    lower_bound = true_mean - q0
    upper_bound = true_mean + q0
    return lower_bound, upper_bound

### 검증: q0가 맞게 계산되는지 두 가지 방법으로 확인

1. **homo + 대칭분포(gaussian, student_t)에서는 $q_0=$ true 분위수와 정확히 같아야 함** — $\sigma_0\equiv1$이면 $R_0=|\varepsilon|$이라서, $q_{0,\alpha}$는 그냥 $\varepsilon$의 $(1-\alpha)$ 분위수. CLAUDE.md §11 "대칭·등분산에서 Oracle=res_true" 검증
2. **hetero는 독립적인 대량 Monte Carlo 표본과 비교** — 적분/root-finding 결과가 실제 표본 분위수와 일치하는지 확인

In [15]:
# 검증 1) homo에서는 q0 == error.ppf(1-alpha/2) 이어야 함 (대칭분포일 때).
#          이유: sigma0_homo(x)=1이라서 R0=|Y-m0(X)|=|eps| 그 자체이고,
#          대칭분포는 상위 (1-alpha/2) 분위수와 -하위(alpha/2) 분위수가 같으므로
#          |eps|의 (1-alpha) 분위수는 그냥 eps의 (1-alpha/2) 분위수와 같아짐.
print("[homo 검증] q0 vs true ppf(1-alpha/2)")
for error_name in ["gaussian", "student_t"]:
    error = ERROR_DISTRIBUTIONS[error_name]
    for alpha in [0.10, 0.05, 0.01]:
        q0 = compute_q0("homo", error_name, alpha)
        q_true = error.ppf(1 - alpha / 2)
        diff = abs(q0 - q_true)
        print(f"  {error_name:10s} alpha={alpha:<5} q0={q0:.6f}  true={q_true:.6f}  diff={diff:.2e}")

# 검증 2) hetero는 이론값이 따로 없으므로, 독립적으로 뽑은 대량 MC 표본의 분위수와 비교.
#          적분+root-finding으로 구한 q0와, 실제 표본에서 직접 잰 분위수가 비슷해야 함.
print("\n[hetero 검증] q0 vs 독립 MC 표본 분위수 (n=20,000,000)")

n_mc = 20_000_000
# 검증 전용 rng로 X1을 뽑음 (본 실험의 데이터 생성 rng와 완전히 분리된 스트림)
x1_check = draw_X(n_mc, substream("validation", "q0_hetero"))[:, 0]  # X2는 sigma0_hetero가 안 쓰므로 무시
X_check_2d = np.column_stack([x1_check, np.zeros(n_mc)])  # sigma0_hetero(X)가 (n,2) 모양을 기대하므로 맞춰줌
sigma0_check = sigma0_hetero(X_check_2d)

for error_name in ["gaussian", "student_t", "lognormal"]:
    error = ERROR_DISTRIBUTIONS[error_name]
    eps_check = error.sample(n_mc, substream("validation", "q0_hetero", error_name))
    R0_check = np.abs(sigma0_check * eps_check)  # R0 = |Y - m0(X)| = |sigma0(X) * eps|

    for alpha in [0.10, 0.05, 0.01]:
        q0_from_integral = compute_q0("hetero", error_name, alpha)
        q0_from_mc_sample = np.quantile(R0_check, 1 - alpha)
        diff = abs(q0_from_integral - q0_from_mc_sample)
        print(
            f"  {error_name:10s} alpha={alpha:<5} "
            f"q0(적분)={q0_from_integral:.6f}  q0(MC)={q0_from_mc_sample:.6f}  diff={diff:.2e}"
        )

[homo 검증] q0 vs true ppf(1-alpha/2)
  gaussian   alpha=0.1   q0=1.644854  true=1.644854  diff=1.40e-12
  gaussian   alpha=0.05  q0=1.959964  true=1.959964  diff=3.33e-15
  gaussian   alpha=0.01  q0=2.575829  true=2.575829  diff=1.33e-15
  student_t  alpha=0.1   q0=1.358715  true=1.358715  diff=5.34e-13
  student_t  alpha=0.05  q0=1.837386  true=1.837386  diff=8.88e-16
  student_t  alpha=0.01  q0=3.372251  true=3.372251  diff=2.04e-11

[hetero 검증] q0 vs 독립 MC 표본 분위수 (n=20,000,000)
  gaussian   alpha=0.1   q0(적분)=1.818114  q0(MC)=1.817921  diff=1.94e-04
  gaussian   alpha=0.05  q0(적분)=2.310716  q0(MC)=2.310791  diff=7.51e-05
  gaussian   alpha=0.01  q0(적분)=3.314805  q0(MC)=3.315089  diff=2.84e-04
  student_t  alpha=0.1   q0(적분)=1.443203  q0(MC)=1.443823  diff=6.20e-04
  student_t  alpha=0.05  q0(적분)=2.017706  q0(MC)=2.017485  diff=2.21e-04
  student_t  alpha=0.01  q0(적분)=3.831315  q0(MC)=3.833006  diff=1.69e-03
  lognormal  alpha=0.1   q0(적분)=1.020377  q0(MC)=1.020757  diff=3.81e-04
  lo

**결과 해석**: 표를 보면 $\alpha$가 작을수록(0.01) 극단 분위수라 수렴이 느리고, $\alpha=0.10$은 상대적으로 빠르게 안정됨. Gaussian은 $10^5$ 정도면 세 alpha 모두 오차가 충분히 작아짐. Student-t·LogNormal은 $\alpha=0.01$에서 훨씬 느리게 줄어들어서, $10^6$으로도 부족하고 $10^7$까지 가야 오차가 한 자릿수 더 줄어듦. 연산 시간은 $10^7$도 몇 초면 끝나서 문제 없음.

**$M_{MC}$ 후보 결정에 이 표를 어떻게 쓰는가**: 나중에 structural/fitting/calibration 분해에서 보려는 차이의 크기(예: 0.01~0.05 수준)보다 이 MC 오차가 충분히 작아야 함. 표에서 그 기준을 넘기는 가장 작은 $M_{MC}$를 고르면 됨 — 지금 수치로는 **$M_{MC}=10^7$** 정도가 가장 어려운 경우(Student-t·LogNormal, $\alpha=0.01$)에서도 안전해 보임 (pilot 제안값, 최종 확정 아님).

## 7. `pop` 구간 ($q_{pop,j,\alpha}$: fitted model 기준 population residual quantile)

Training으로 학습한 fitted model $\widehat m_j$를 고정하고, **fitted model 기준 residual** $R_j=|Y-\widehat m_j(X)|$의 population quantile로 폭을 정함:

$$C_{pop,j}(x) = [\widehat m_j(x)-q_{pop,j,\alpha},\ \widehat m_j(x)+q_{pop,j,\alpha}]$$

`res_true`(§6)와 식은 똑같지만($g=m_0$ 대신 $g=\widehat m_j$), 계산 방법이 다름:
- `res_true`는 $\sigma_0(X)$가 $X_1$에만 의존해서 1차원 적분으로 정확히 계산 가능
- `pop`은 $\widehat m_j(X)$가 $X_1,X_2$ 둘 다에 의존하므로, 적분으로 직접 계산하기가 훨씬 까다로움 -> $$ X_1^*,\ldots,X_{M_{\text{pop}}}^* \sim P_X $$를 많이 뽑아서 $X$에 대한 평균을 Monte Carlo로 근사->
**독립적인 reference 입력 표본**(`M_pop`개)으로 $H_g(q)=E_X[p_g(q,X)]$를 몬테카를로 근사 (CLAUDE.md §5)

Reference 입력은 training·calibration·평가 입력과 독립이고, **같은 fitted model이면 OLS·RF가 이 reference를 공유** — 모델 예측값만 다르고 reference 위치는 같음.

In [16]:
def draw_population_reference(M_pop, rng):
    """pop 구간용 독립 reference 입력. training·calibration·평가 입력과 무관하며,
    같은 (dgp, b)에서 OLS·RF가 이 표본을 공유한다."""
    return draw_X(M_pop, rng)


def _H_pop(q, cached_g, cached_true_mean, cached_true_scale, error):
    """H_g(q) = E_X[p_g(q,X)]를, 미리 계산해둔(cached) reference 값들로 근사.

    cached_g, cached_true_mean, cached_true_scale은 reference 입력에서 한 번만 계산해두고
    q를 바꿔가며(root-finding) 여러 번 재사용 -> fitted_model.predict를 반복 호출하지 않음
    (CLAUDE.md §9: 예측은 동일 입력에서 캐시하고 재사용).
    """
    upper_z = (cached_g + q - cached_true_mean) / cached_true_scale
    lower_z = (cached_g - q - cached_true_mean) / cached_true_scale
    p_values = error.cdf(upper_z) - error.cdf(lower_z)  # reference 각 점에서의 "포함 확률"
    return p_values.mean()  # 전체 reference에 대한 평균 = H_g(q)의 근사값


def compute_q_pop(fitted_model, ref_X, dgp, alpha):
    """H_g(q) = 1-alpha 를 만족하는 q_pop을 찾는다 (pop 구간의 반폭)."""
    # 1단계: reference 입력에서 필요한 값들을 딱 한 번만 계산해서 캐시
    g_at_ref = fitted_model.predict(ref_X)  # g(x) = fitted model의 예측값
    true_mean_at_ref = dgp.mean_fn(ref_X)
    true_scale_at_ref = dgp.scale_fn(ref_X)

    # 2단계: 그 캐시를 이용해 H(q)-target = 0 이 되는 q를 root-finding으로 찾음 (res_true와 같은 방식)
    target_probability = 1 - alpha

    def h_minus_target(q):
        return _H_pop(q, g_at_ref, true_mean_at_ref, true_scale_at_ref, dgp.error) - target_probability

    return optimize.brentq(h_minus_target, 1e-8, 50.0, xtol=1e-10)


def pop_interval(X, fitted_model, q_pop):
    """[g(x) - q_pop, g(x) + q_pop]. fitted model의 예측값을 중심으로, 폭은 q_pop으로 고정."""
    g = fitted_model.predict(X)
    return g - q_pop, g + q_pop

### 확인: q_pop vs q0, 그리고 M_pop 안정성

§4에서 이미 학습해둔 `linear_homo_gaussian`의 `ols_model`, `rf_model`(둘 다 b=1)을 그대로 재사용. 대칭분포(gaussian)이므로 CLAUDE.md §11 "fitting half-width는 비음수" 검증대로 $q_{pop}\ge q_0$이어야 함 — OLS는 이 DGP에서 정확히 맞는 모형이라 $q_0$에 가까울 것, RF는 유한표본 추정오차 때문에 더 클 것으로 예상.

In [17]:
dgp = DGPS["linear_homo_gaussian"]  # ols_model, rf_model은 §4에서 이 dgp의 b=1로 이미 학습해둔 것
alpha = 0.05
b = 1

q0 = compute_q0(dgp.scale_name, dgp.error_name, alpha)  # res_true 반폭 (비교 기준)
print(f"q0 (res_true) = {q0:.5f}\n")

# M_pop을 늘려가면서 q_pop이 안정적으로 수렴하는지 확인 (같은 M_pop이라도 다른 reference로 3번 반복)
print(f"{'M_pop':>8} | {'q_pop(OLS) 평균':>16} {'sd':>10} | {'q_pop(RF) 평균':>16} {'sd':>10}")
for M_pop in [2_000, 10_000, 20_000]:
    q_pop_ols_reps, q_pop_rf_reps = [], []
    for rep in range(3):
        rng_reference = substream(dgp.name, b, "reference", "M_pop_check", M_pop, rep)
        ref_X = draw_population_reference(M_pop, rng_reference)
        q_pop_ols_reps.append(compute_q_pop(ols_model, ref_X, dgp, alpha))
        q_pop_rf_reps.append(compute_q_pop(rf_model, ref_X, dgp, alpha))
    print(
        f"{M_pop:>8,} | {np.mean(q_pop_ols_reps):>16.5f} {np.std(q_pop_ols_reps):>10.2e} | "
        f"{np.mean(q_pop_rf_reps):>16.5f} {np.std(q_pop_rf_reps):>10.2e}"
    )

# 최종적으로 M_pop=20,000, 이 training rep(b=1)의 "진짜" reference 스트림으로 q_pop 확정
rng_reference = substream(dgp.name, b, "reference")
ref_X = draw_population_reference(20_000, rng_reference)
q_pop_ols = compute_q_pop(ols_model, ref_X, dgp, alpha)
q_pop_rf = compute_q_pop(rf_model, ref_X, dgp, alpha)
print(f"\nq_pop(OLS) = {q_pop_ols:.5f}  (q_pop - q0 = {q_pop_ols - q0:+.5f}, >= 0 이어야 함)")
print(f"q_pop(RF)  = {q_pop_rf:.5f}  (q_pop - q0 = {q_pop_rf - q0:+.5f}, >= 0 이어야 함)")

# ---- 여기까지는 "폭"(q_pop)만 구한 것. 실제 구간 C_pop,j(x) = [g(x)-q_pop, g(x)+q_pop]을 만들어보면 ----
x_eval = np.array([[0.0, 0.0], [0.5, -0.5], [0.9, 0.9]])
L_pop_ols, U_pop_ols = pop_interval(x_eval, ols_model, q_pop_ols)
L_pop_rf, U_pop_rf = pop_interval(x_eval, rf_model, q_pop_rf)

print("\n실제 C_pop,j(x) 구간 (x_eval의 각 행마다 [L, U]):")
print("pop(OLS):", np.stack([L_pop_ols, U_pop_ols], axis=1))
print("pop(RF) :", np.stack([L_pop_rf, U_pop_rf], axis=1))

# 참고: res_true 구간과 중심 비교 (res_true는 true m0를 중심으로, pop은 fitted model 예측값을 중심으로 함)
L_res, U_res = res_true_interval(x_eval, dgp, alpha)
print("res_true:", np.stack([L_res, U_res], axis=1))

q0 (res_true) = 1.95996

   M_pop |    q_pop(OLS) 평균         sd |     q_pop(RF) 평균         sd
   2,000 |          1.96232   6.47e-05 |          2.14028   9.15e-03
  10,000 |          1.96233   1.32e-05 |          2.14277   2.20e-03
  20,000 |          1.96234   2.21e-05 |          2.14071   1.43e-03

q_pop(OLS) = 1.96230  (q_pop - q0 = +0.00234, >= 0 이어야 함)
q_pop(RF)  = 2.13830  (q_pop - q0 = +0.17834, >= 0 이어야 함)

실제 C_pop,j(x) 구간 (x_eval의 각 행마다 [L, U]):
pop(OLS): [[-1.978001  1.946604]
 [-2.033498  1.891108]
 [ 0.542969  4.467574]]
pop(RF) : [[-2.44826   1.828338]
 [-2.657183  1.619416]
 [ 0.249392  4.52599 ]]
res_true: [[-1.959964  1.959964]
 [-1.959964  1.959964]
 [ 0.58562   4.505548]]


## 8. `cp` 구간 (실제 finite-sample Split Conformal Prediction)

앞의 `pop`은 "calibration이 무한히 많다"는 가정이었는데, `cp`는 **실제 유한한 calibration 데이터 $m$개**로 만드는 진짜 conformal 구간입니다:

$$C_{CP,j}(x) = [\widehat m_j(x)-\widehat q_{j,\alpha},\ \widehat m_j(x)+\widehat q_{j,\alpha}]$$

$\widehat q_{j,\alpha}$를 구하는 방법은 root-finding이 아니라 훨씬 단순합니다:

1. calibration $m$개의 절대잔차 $R_i=|Y_i-\widehat m_j(X_i)|$를 오름차순 정렬
2. 순위 $k_\alpha=\lceil (m+1)(1-\alpha)\rceil$ 계산
3. $k_\alpha\le m$이면 정렬한 배열의 $k_\alpha$번째(1-indexed) 값 사용, $k_\alpha=m+1$이면 $+\infty$

$m=1000$이면 순위는 alpha=[0.10,0.05,0.01]에 대해 정확히 **901, 951, 991**이어야 합니다 (CLAUDE.md §4, §11 필수 검증 항목). numpy 기본 quantile 보간은 이 규칙과 다르므로 절대 쓰지 않고, 정렬된 배열에서 정확한 순서통계량을 직접 뽑습니다.

In [18]:
def cp_quantile(fitted_model, X_cal, y_cal, alpha):
    """실제 calibration 데이터로 conformal quantile(반폭) q_hat을 계산.

    반환값: (q_hat, info). info에는 순위 k, calibration 크기 m, 상태(finite/infinite)를 담아서
    k=m+1(무한 구간)인지 나중에 명확히 구분할 수 있게 함.
    """
    # 1단계: 절대잔차 계산 (같은 fitted model이라도 pop과 달리 실제 데이터 m개만 사용)
    residuals = np.abs(y_cal - fitted_model.predict(X_cal))
    sorted_residuals = np.sort(residuals)  # 오름차순 정렬
    m = len(sorted_residuals)

    # 2단계: conformal 순위 계산. -1e-9는 부동소수점 오차로 (m+1)(1-alpha)가
    # 정확히 정수여야 하는 경계에서 한 단계 밀리는 걸 막기 위한 안전장치
    k = math.ceil((m + 1) * (1 - alpha) - 1e-9)

    # 3단계: k가 calibration 크기를 넘으면(=m+1) 무한 구간, 아니면 k번째(1-indexed) 값 사용
    if k > m:
        q_hat = np.inf
        state = "infinite"
    else:
        q_hat = sorted_residuals[k - 1]  # 1-indexed k번째 = 0-indexed (k-1)번째
        state = "finite"

    info = {"k": k, "m": m, "state": state}
    return q_hat, info


def cp_interval(X, fitted_model, q_hat):
    """[g(x) - q_hat, g(x) + q_hat]. pop_interval과 형태는 같고 폭(q_hat)만 다른 방식으로 구한 것."""
    g = fitted_model.predict(X)
    return g - q_hat, g + q_hat

### 검증: 순위 901/951/991, 그리고 $k=m+1$ 경계

CLAUDE.md §11 필수 검증 항목. 알려진 배열(1,2,...,m)로 순위가 정확한지 확인하고, calibration이 작고 alpha가 작아서 $k$가 $m$을 넘는 경우(무한 구간)도 확인.

In [19]:
class _ZeroModel:
    """검증용 가짜 모델: 항상 0을 예측 -> residual = |y - 0| = y 그대로 되어 known array를 검증에 쓸 수 있음."""

    def predict(self, X):
        return np.zeros(len(X))


_zero_model = _ZeroModel()

# 1) m=1000, 알려진 배열 [1,2,...,1000]으로 순위가 정확히 901/951/991인지 확인
known_residuals = np.arange(1, 1001, dtype=float)
X_dummy = np.zeros((1000, 2))
print("[m=1000 순위 검증]")
for alpha, expected_k in zip([0.10, 0.05, 0.01], [901, 951, 991]):
    q_hat, info = cp_quantile(_zero_model, X_dummy, known_residuals, alpha)
    print(f"  alpha={alpha:<5} k={info['k']} (target {expected_k})  q_hat={q_hat} (target {expected_k})")
    assert info["k"] == expected_k, f"순위 불일치: {info['k']} != {expected_k}"
    assert q_hat == expected_k, f"q_hat 불일치: {q_hat} != {expected_k}"

# 2) k = m+1 경계: calibration이 작고 alpha가 아주 작으면 무한 구간이 나와야 함
print("\n[k=m+1 경계 검증]")
small_residuals = np.arange(1, 11, dtype=float)  # m=10
X_dummy_small = np.zeros((10, 2))
q_hat, info = cp_quantile(_zero_model, X_dummy_small, small_residuals, alpha=0.01)
print(f"  m=10, alpha=0.01: k={info['k']}, m={info['m']}, q_hat={q_hat}, state={info['state']}")
assert info["state"] == "infinite" and np.isinf(q_hat)

print("\nCP 순위·경계 검증 통과")

[m=1000 순위 검증]
  alpha=0.1   k=901 (target 901)  q_hat=901.0 (target 901)
  alpha=0.05  k=951 (target 951)  q_hat=951.0 (target 951)
  alpha=0.01  k=991 (target 991)  q_hat=991.0 (target 991)

[k=m+1 경계 검증]
  m=10, alpha=0.01: k=11, m=10, q_hat=inf, state=infinite

CP 순위·경계 검증 통과


### 확인: 실제 calibration으로 cp 구간 만들기, pop과 비교

같은 `linear_homo_gaussian`의 `ols_model`/`rf_model`(b=1)에, 실제 독립 calibration $m=1000$개를 만들어 cp 구간을 계산. `cp`는 유한 calibration을 쓰므로 `pop`(calibration 무한 가정)보다 표본 변동이 있어야 하지만, $m=1000$이면 비슷한 값이 나올 것으로 예상.

In [20]:
dgp = DGPS["linear_homo_gaussian"]  # ols_model, rf_model은 §4에서 이 dgp의 b=1로 학습해둔 것
b, r = 1, 1
alpha = 0.05

# 실제 독립 calibration 데이터 (같은 (b, r)에서 OLS, RF가 이 calibration을 공유)
rng_cal = substream(dgp.name, b, r, "calibration")
X_cal, y_cal = dgp.sample_xy(1000, rng_cal)

q_hat_ols, info_ols = cp_quantile(ols_model, X_cal, y_cal, alpha)
q_hat_rf, info_rf = cp_quantile(rf_model, X_cal, y_cal, alpha)
print(f"cp: q_hat(OLS)={q_hat_ols:.5f}  {info_ols}")
print(f"cp: q_hat(RF) ={q_hat_rf:.5f}  {info_rf}")

# 앞서(§7) 계산해둔 q_pop과 비교 -- 값이 유한 calibration 표본변동만큼만 다르면 정상
print(f"\npop: q_pop(OLS)={q_pop_ols:.5f}   cp - pop = {q_hat_ols - q_pop_ols:+.5f}")
print(f"pop: q_pop(RF) ={q_pop_rf:.5f}   cp - pop = {q_hat_rf - q_pop_rf:+.5f}")

# 실제 구간까지 만들어서 확인
x_eval = np.array([[0.0, 0.0], [0.5, -0.5], [0.9, 0.9]])
L_cp_ols, U_cp_ols = cp_interval(x_eval, ols_model, q_hat_ols)
L_cp_rf, U_cp_rf = cp_interval(x_eval, rf_model, q_hat_rf)
print("\ncp(OLS):", np.stack([L_cp_ols, U_cp_ols], axis=1))
print("cp(RF) :", np.stack([L_cp_rf, U_cp_rf], axis=1))

cp: q_hat(OLS)=1.97794  {'k': 951, 'm': 1000, 'state': 'finite'}
cp: q_hat(RF) =2.12029  {'k': 951, 'm': 1000, 'state': 'finite'}

pop: q_pop(OLS)=1.96230   cp - pop = +0.01563
pop: q_pop(RF) =2.13830   cp - pop = -0.01800

cp(OLS): [[-1.993635  1.962237]
 [-2.049131  1.906741]
 [ 0.527335  4.483207]]
cp(RF) : [[-2.430256  1.810334]
 [-2.639178  1.601412]
 [ 0.267396  4.507986]]


## 9. 다섯 구간 한눈에 비교하는 표

`linear_homo_gaussian`, $\alpha=0.05$, `x_eval`의 세 위치에서 다섯 구간(oracle·mc·res_true·pop·cp)을 전부 계산해서 하나의 표로 정리. `oracle`/`mc`/`res_true`는 모델과 무관하므로 `model_id="-"`, `pop`/`cp`는 OLS·RF 각각.

In [21]:
dgp = DGPS["linear_homo_gaussian"]  # ols_model, rf_model, q0, q_pop_*, q_hat_*는 위에서 이미 계산해둔 값들
alpha = 0.05
x_eval = np.array([[0.0, 0.0], [0.5, -0.5], [0.9, 0.9]])

# mc만 이 dgp(error=gaussian)에 맞춰 새로 뽑음 (앞선 mc 데모는 다른 dgp였으므로)
eps_sample_gaussian = draw_mc_benchmark(dgp, 1_000_000, substream("mc_benchmark", dgp.error_name))

# (interval_id, model_id, (L, U)) 튜플을 하나씩 나열 -- model_id가 없는(None) 구간은 표에서 "-"로 표시
families = [
    ("oracle", None, oracle_interval(x_eval, dgp, alpha)),
    ("mc", None, mc_interval(x_eval, dgp, alpha, eps_sample_gaussian)),
    ("res_true", None, res_true_interval(x_eval, dgp, alpha)),
    ("pop", "OLS", pop_interval(x_eval, ols_model, q_pop_ols)),
    ("pop", "RF", pop_interval(x_eval, rf_model, q_pop_rf)),
    ("cp", "OLS", cp_interval(x_eval, ols_model, q_hat_ols)),
    ("cp", "RF", cp_interval(x_eval, rf_model, q_hat_rf)),
]

rows = []
for interval_id, model_id, (L, U) in families:
    coverage = conditional_coverage(L, U, x_eval, dgp)
    for i in range(len(x_eval)):
        rows.append({
            "interval_id": interval_id,
            "model_id": model_id or "-",
            "x1": x_eval[i, 0],
            "x2": x_eval[i, 1],
            "L": L[i],
            "U": U[i],
            "center": (L[i] + U[i]) / 2,
            "half_width": (U[i] - L[i]) / 2,
            "coverage": coverage[i],
        })

five_intervals_table = pd.DataFrame(rows)
five_intervals_table

,interval_id,model_id,x1,x2,L,U,center,half_width,coverage
0,oracle,-,0.0000,0.0000,-1.9600,1.9600,-0.0000,1.9600,0.9500
1,oracle,-,0.5000,-0.5000,-1.9600,1.9600,-0.0000,1.9600,0.9500
2,oracle,-,0.9000,0.9000,0.5856,4.5055,2.5456,1.9600,0.9500
3,mc,-,0.0000,0.0000,-1.9583,1.9611,0.0014,1.9597,0.9500
4,mc,-,0.5000,-0.5000,-1.9583,1.9611,0.0014,1.9597,0.9500
5,mc,-,0.9000,0.9000,0.5873,4.5067,2.5470,1.9597,0.9500
6,res_true,-,0.0000,0.0000,-1.9600,1.9600,0.0000,1.9600,0.9500
7,res_true,-,0.5000,-0.5000,-1.9600,1.9600,0.0000,1.9600,0.9500
8,res_true,-,0.9000,0.9000,0.5856,4.5055,2.5456,1.9600,0.9500
9,pop,OLS,0.0000,0.0000,-1.9780,1.9466,-0.0157,1.9623,0.9502


### 같은 표를 pivot으로 정리 (지표별로 한눈에 비교)

`interval_id`+`model_id`를 합쳐 하나의 열 이름(`family`, 예: `pop_RF`)으로 만들고, 위치(x1,x2)를 행으로 두고 각 지표(center, half_width, coverage)마다 별도의 작은 표를 만든다. 열 순서는 oracle→mc→res_true→pop_OLS→pop_RF→cp_OLS→cp_RF로 고정해서 원래 5개 구간 순서를 그대로 따라가게 함.

In [22]:
from IPython.display import display

FAMILY_ORDER = ["oracle", "mc", "res_true", "pop_OLS", "pop_RF", "cp_OLS", "cp_RF"]

# interval_id와 model_id를 합쳐서 "family" 하나의 이름으로 만듦 (model_id가 "-"면 그냥 interval_id만 씀)
table_with_family = five_intervals_table.copy()
table_with_family["family"] = table_with_family.apply(
    lambda row: row["interval_id"] if row["model_id"] == "-" else f"{row['interval_id']}_{row['model_id']}",
    axis=1,
)

# 지표(center, half_width, coverage)마다 따로 pivot 표를 만들고, 각각을 별도의 표로 보여줌
center_table = table_with_family.pivot(index=["x1", "x2"], columns="family", values="center")[FAMILY_ORDER]
half_width_table = table_with_family.pivot(index=["x1", "x2"], columns="family", values="half_width")[FAMILY_ORDER]
coverage_table = table_with_family.pivot(index=["x1", "x2"], columns="family", values="coverage")[FAMILY_ORDER]

print("=== center ===")
display(center_table)

print("\n=== half_width ===")
display(half_width_table)

print("\n=== coverage (target 0.95) ===")
display(coverage_table)

=== center ===


,family,oracle,mc,res_true,pop_OLS,pop_RF,cp_OLS,cp_RF
x1,x2,,,,,,,
0.0000,0.0000,-0.0000,0.0014,0.0000,-0.0157,-0.3100,-0.0157,-0.3100
0.5000,-0.5000,-0.0000,0.0014,0.0000,-0.0712,-0.5189,-0.0712,-0.5189
0.9000,0.9000,2.5456,2.5470,2.5456,2.5053,2.3877,2.5053,2.3877



=== half_width ===


,family,oracle,mc,res_true,pop_OLS,pop_RF,cp_OLS,cp_RF
x1,x2,,,,,,,
0.0000,0.0000,1.9600,1.9597,1.9600,1.9623,2.1383,1.9779,2.1203
0.5000,-0.5000,1.9600,1.9597,1.9600,1.9623,2.1383,1.9779,2.1203
0.9000,0.9000,1.9600,1.9597,1.9600,1.9623,2.1383,1.9779,2.1203



=== coverage (target 0.95) ===


,family,oracle,mc,res_true,pop_OLS,pop_RF,cp_OLS,cp_RF
x1,x2,,,,,,,
0.0000,0.0000,0.9500,0.9500,0.9500,0.9502,0.9591,0.9520,0.9573
0.5000,-0.5000,0.9500,0.9500,0.9500,0.9497,0.9434,0.9515,0.9412
0.9000,0.9000,0.9500,0.9500,0.9500,0.9501,0.9653,0.9519,0.9638


### 진짜 구간 $[L, U]$ 형태로 보기

앞의 표들은 center·half_width·coverage처럼 구간에서 계산한 숫자만 보여줬는데, 실제 구간 끝점을 `[L, U]` 문자열로 그대로 보여주는 표.

In [23]:
# L, U를 "[L, U]" 형태의 문자열로 합쳐서, family별로 나란히 볼 수 있는 표를 만듦
table_with_family["interval_str"] = table_with_family.apply(
    lambda row: f"[{row['L']:.4f}, {row['U']:.4f}]", axis=1
)

interval_bracket_table = table_with_family.pivot(index=["x1", "x2"], columns="family", values="interval_str")
interval_bracket_table = interval_bracket_table[FAMILY_ORDER]  # 열 순서를 5개 구간의 원래 순서로 고정
interval_bracket_table

,family,oracle,mc,res_true,pop_OLS,pop_RF,cp_OLS,cp_RF
x1,x2,,,,,,,
0.0000,0.0000,"[-1.9600, 1.9600]","[-1.9583, 1.9611]","[-1.9600, 1.9600]","[-1.9780, 1.9466]","[-2.4483, 1.8283]","[-1.9936, 1.9622]","[-2.4303, 1.8103]"
0.5000,-0.5000,"[-1.9600, 1.9600]","[-1.9583, 1.9611]","[-1.9600, 1.9600]","[-2.0335, 1.8911]","[-2.6572, 1.6194]","[-2.0491, 1.9067]","[-2.6392, 1.6014]"
0.9000,0.9000,"[0.5856, 4.5055]","[0.5873, 4.5067]","[0.5856, 4.5055]","[0.5430, 4.4676]","[0.2494, 4.5260]","[0.5273, 4.4832]","[0.2674, 4.5080]"


## 10. Structural / Fitting / Calibration 분해

CP−MC의 전체 차이(`total_mc`)를 Oracle을 경유해서 4항으로 telescoping 분해:

| component | 정의 |
| --- | --- |
| `structural` | $T(C_{res,0})-T(C_{Oracle})$ |
| `fitting` | $T(C_{pop,j})-T(C_{res,0})$ |
| `calibration` | $T(C_{CP,j})-T(C_{pop,j})$ |
| `mc_approximation` | $T(C_{Oracle})-T(C_{MC})$ |

$$\text{total\_mc} = \text{structural} + \text{fitting} + \text{calibration} + \text{mc\_approximation}$$

**signed difference**이며 절댓값을 취하지 않음 — 항끼리 상쇄 가능. `coverage`, `center`, `half_width`, `length`(=2×half_width) 네 지표 모두에 대해 계산 (CLAUDE.md §6 필수 4개 지표).

In [24]:
def decompose(T):
    """다섯 구간의 같은 지표 값을 signed telescoping으로 4항 분해.

    T: {"oracle":.., "mc":.., "res_true":.., "pop":.., "cp":..} 형태의 dict (같은 위치·같은 지표의 값들).
    절댓값을 취하지 않으므로 항끼리 상쇄될 수 있음 -- 독립적인 인과 기여율이 아니라 signed 분해.
    """
    structural = T["res_true"] - T["oracle"]        # true mean은 알아도 하나의 population quantile로 폭을 정하며 생기는 차이
    fitting = T["pop"] - T["res_true"]              # true mean 대신 fitted model을 쓰면서 생기는 차이
    calibration = T["cp"] - T["pop"]                # population(무한 calibration) 대신 유한 calibration을 쓰면서 생기는 차이
    mc_approximation = T["oracle"] - T["mc"]        # true quantile 대신 유한 MC 표본으로 근사하며 생기는 차이
    total_mc = T["cp"] - T["mc"]                    # 실제 비교 대상인 CP - MC 전체 차이

    # 네 항을 더하면 total_mc와 정확히 같아야 함 (부동소수점 오차 수준 제외)
    identity_residual = total_mc - (structural + fitting + calibration + mc_approximation)

    return {
        "total_mc": total_mc,
        "structural": structural,
        "fitting": fitting,
        "calibration": calibration,
        "mc_approximation": mc_approximation,
        "identity_residual": identity_residual,
    }


# §9에서 만든 table_with_family를 (family, x1, x2) -> 행으로 바로 찾아볼 수 있게 인덱싱
lookup = table_with_family.set_index(["family", "x1", "x2"])


def get_metric(family, x1, x2, metric):
    return lookup.loc[(family, x1, x2), metric]


decomposition_rows = []
for model_id in ["OLS", "RF"]:
    pop_family = f"pop_{model_id}"
    cp_family = f"cp_{model_id}"

    for x1, x2 in x_eval:
        for metric in ["coverage", "center", "half_width", "length"]:
            if metric == "length":
                # length = 2 * half_width. half_width의 다섯 값을 각각 2배 하면 length 기준 분해가 됨
                T = {
                    "oracle": 2 * get_metric("oracle", x1, x2, "half_width"),
                    "mc": 2 * get_metric("mc", x1, x2, "half_width"),
                    "res_true": 2 * get_metric("res_true", x1, x2, "half_width"),
                    "pop": 2 * get_metric(pop_family, x1, x2, "half_width"),
                    "cp": 2 * get_metric(cp_family, x1, x2, "half_width"),
                }
            else:
                T = {
                    "oracle": get_metric("oracle", x1, x2, metric),
                    "mc": get_metric("mc", x1, x2, metric),
                    "res_true": get_metric("res_true", x1, x2, metric),
                    "pop": get_metric(pop_family, x1, x2, metric),
                    "cp": get_metric(cp_family, x1, x2, metric),
                }

            components = decompose(T)
            # T(원래 다섯 구간 값)와 components(분해 결과)를 한 행에 같이 저장 -> 나중에 나란히 볼 수 있게
            decomposition_rows.append({
                "model_id": model_id, "x1": x1, "x2": x2, "metric": metric,
                **T, **components,
            })

decomposition_table = pd.DataFrame(decomposition_rows)

# 등식 검증: identity_residual이 전부 0에 가까워야 함 (부동소수점 오차 수준)
max_residual = decomposition_table["identity_residual"].abs().max()
print(f"max |identity_residual| = {max_residual:.2e} (거의 0이어야 함)")

# center의 calibration 항은 항상 0이어야 함 (같은 fitted model이면 pop/cp의 중심이 같으므로)
center_calibration = decomposition_table.query("metric == 'center'")["calibration"]
print(f"center의 calibration 항 최대 절댓값 = {center_calibration.abs().max():.2e} (0이어야 함)")

max |identity_residual| = 0.00e+00 (거의 0이어야 함)
center의 calibration 항 최대 절댓값 = 0.00e+00 (0이어야 함)


In [25]:
# 왼쪽 5개 열 = 실제 다섯 구간의 값(T), 오른쪽 5개 열 = 그걸로 계산한 분해 결과.
# 분해 숫자만 보면 "그래서 원래 구간 값이 뭐였는데?"를 알 수 없으므로 항상 같이 보여줌.
RAW_ORDER = ["oracle", "mc", "res_true", "pop", "cp"]
COMPONENT_ORDER = ["total_mc", "structural", "fitting", "calibration", "mc_approximation"]
METRIC_ORDER = ["coverage", "center", "half_width", "length"]

for model_id in ["OLS", "RF"]:
    print(f"\n{'=' * 10} model = {model_id} {'=' * 10}")
    model_rows = decomposition_table[decomposition_table["model_id"] == model_id]

    for metric in METRIC_ORDER:
        print(f"--- {metric} ---")
        metric_rows = model_rows[model_rows["metric"] == metric]
        pivoted = metric_rows.set_index(["x1", "x2"])[RAW_ORDER + COMPONENT_ORDER]
        display(pivoted)


========== model = OLS ==========
--- coverage ---


,,oracle,mc,res_true,pop,cp,total_mc,structural,fitting,calibration,mc_approximation
x1,x2,,,,,,,,,,
0.0000,0.0000,0.9500,0.9500,0.9500,0.9502,0.9520,0.0021,-0.0000,0.0002,0.0018,0.0000
0.5000,-0.5000,0.9500,0.9500,0.9500,0.9497,0.9515,0.0015,-0.0000,-0.0003,0.0018,0.0000
0.9000,0.9000,0.9500,0.9500,0.9500,0.9501,0.9519,0.0019,-0.0000,0.0001,0.0018,0.0000


--- center ---


,,oracle,mc,res_true,pop,cp,total_mc,structural,fitting,calibration,mc_approximation
x1,x2,,,,,,,,,,
0.0000,0.0000,-0.0000,0.0014,0.0000,-0.0157,-0.0157,-0.0171,0.0000,-0.0157,0.0000,-0.0014
0.5000,-0.5000,-0.0000,0.0014,0.0000,-0.0712,-0.0712,-0.0726,0.0000,-0.0712,0.0000,-0.0014
0.9000,0.9000,2.5456,2.5470,2.5456,2.5053,2.5053,-0.0417,0.0000,-0.0403,0.0000,-0.0014


--- half_width ---


...

,,oracle,mc,res_true,pop,cp,total_mc,structural,fitting,calibration,mc_approximation
x1,x2,,,,,,,,,,
0.0000,0.0000,-0.0000,0.0014,0.0000,-0.3100,-0.3100,-0.3114,0.0000,-0.3100,0.0000,-0.0014
0.5000,-0.5000,-0.0000,0.0014,0.0000,-0.5189,-0.5189,-0.5203,0.0000,-0.5189,0.0000,-0.0014
0.9000,0.9000,2.5456,2.5470,2.5456,2.3877,2.3877,-0.1593,0.0000,-0.1579,0.0000,-0.0014


--- half_width ---


,,oracle,mc,res_true,pop,cp,total_mc,structural,fitting,calibration,mc_approximation
x1,x2,,,,,,,,,,
0.0000,0.0000,1.9600,1.9597,1.9600,2.1383,2.1203,0.1606,-0.0000,0.1783,-0.0180,0.0003
0.5000,-0.5000,1.9600,1.9597,1.9600,2.1383,2.1203,0.1606,-0.0000,0.1783,-0.0180,0.0003
0.9000,0.9000,1.9600,1.9597,1.9600,2.1383,2.1203,0.1606,-0.0000,0.1783,-0.0180,0.0003


--- length ---


,,oracle,mc,res_true,pop,cp,total_mc,structural,fitting,calibration,mc_approximation
x1,x2,,,,,,,,,,
0.0000,0.0000,3.9199,3.9194,3.9199,4.2766,4.2406,0.3212,-0.0000,0.3567,-0.0360,0.0005
0.5000,-0.5000,3.9199,3.9194,3.9199,4.2766,4.2406,0.3212,-0.0000,0.3567,-0.0360,0.0005
0.9000,0.9000,3.9199,3.9194,3.9199,4.2766,4.2406,0.3212,-0.0000,0.3567,-0.0360,0.0005


### 이 표가 의미하는 것

각 표는 "CP와 MC 사이의 전체 차이(`total_mc`)가 어느 단계에서 왜 생기는지"를 네 조각으로 쪼갠 것. 열은 항상 같은 순서(`total_mc → structural → fitting → calibration → mc_approximation`)이고, 이 넷을 더하면 정확히 `total_mc`가 됨(§10에서 `identity_residual=0`으로 이미 확인).

- **`structural`**: true mean $m_0$을 알아도, "위치마다 다른 정확한 분포" 대신 "전체를 뭉쳐서 하나의 폭"으로 정하면서 생기는 차이. 지금 DGP(등분산+대칭)에서는 항상 0 — 위치에 따라 폭이 달라질 이유가 없어서.
- **`fitting`**: true mean 대신 **fitted model**(OLS/RF)을 쓰면서 생기는 차이. 지금 결과를 보면 **RF는 이 항이 크고(예: half_width에서 +0.18), OLS는 거의 0**에 가까움 — DGP가 선형이라 OLS는 거의 정답을 맞히고, RF는 유한표본 추정오차가 크기 때문.
- **`calibration`**: population(무한 calibration 가정) 대신 **실제 유한 calibration**(m=1000)을 쓰면서 생기는 차이. 두 모델 다 `fitting`보다 훨씬 작음 — m=1000이면 이미 충분히 안정적이라는 뜻.
- **`mc_approximation`**: Oracle 대신 **MC의 유한 표본 근사**를 쓰면서 생기는 차이. 이것도 작음 ($M_{MC}=10^6$이 충분히 커서).

**해석 포인트**: 지금 이 DGP·이 위치들에서는 `total_mc`(CP-MC 전체 차이)의 대부분이 **`fitting`에서 나온다**(특히 RF). `calibration`과 `mc_approximation`은 상대적으로 작은 보정 역할. 이건 "RF가 항상 나쁘다"는 뜻이 아니라, **지금 DGP(선형)가 OLS에 유리한 특수 케이스**라서 이런 패턴이 나온 것 — `mean=nonlinear`로 바꾸면 OLS 쪽에도 `fitting`(misspecification) 기여가 생겨서 패턴이 달라질 것으로 예상.

각 항은 **signed(부호 있는) 값**이라 서로 상쇄될 수 있고, 절댓값 합이 전체 차이와 같다는 뜻은 아님 — 어떤 위치에서는 `calibration`이 `fitting`을 살짝 상쇄하는 방향(음수)으로 나타나는 것도 그 예.

## 11. Marginal 관점 ($B=200$)

지금까지는 $b=1$ 하나의 training·calibration에서만 봤는데, 이제 **$b=1,\ldots,200$ 독립 반복**에 걸쳐 평균을 낸다. 각 $b$에서: training 1000개 생성 → OLS·RF 학습 → 독립 calibration 1000개($r=1$) → 독립 평가 입력 $N_{eval}=2000$개에서 다섯 구간·지표·분해를 계산 → 그 2000개 평가 입력에 대해 평균 → 이게 그 $b$ 하나의 요약값. 이 요약값을 $B=200$개 모아서 다시 평균 내고, $\text{MCSE}=\text{SD}/\sqrt B$로 반복 간 변동을 보고한다 (CLAUDE.md §7, §9.2).

$M_{MC}=2{,}000{,}000$, $M_{pop}=20{,}000$, $N_{eval}=2000$은 이번 rough pass의 잠정값 (pilot 정식 검증 전).

계산은 별도 스크립트로 백그라운드 실행해서 `results/marginal_linear_homo_gaussian.csv`에 저장해뒀고, 여기서는 그 결과를 읽어서 집계·표시만 한다.

### 실행 코드 (이미 계산된 결과가 있으면 재계산 생략)

`RUN_MARGINAL_FULL = True`로 바꾸면 처음부터 다시 계산하고, 기본값(`False`)이면 이미 저장된 `results/marginal_linear_homo_gaussian.csv`가 있을 때 재계산을 건너뛴다 — 매번 노트북을 열 때마다 35분씩 다시 돌리지 않기 위함. 아래 함수는 위에서 만든 `oracle_interval`/`mc_interval`/`res_true_interval`/`compute_q_pop`/`pop_interval`/`cp_quantile`/`cp_interval`/`conditional_coverage`/`decompose`를 그대로 재사용 — 새로 정의하는 수식은 없음.

In [ ]:
N_TRAIN, N_CAL, M_POP = 1000, 1000, 20_000
N_EVAL = 2_000  # marginal 평가 입력 수 (pilot 확정 전 잠정값)
ALPHAS = [0.10, 0.05, 0.01]
M_MC_ROUGH = 2_000_000  # 이번 rough pass의 잠정 M_MC (§5 탐색에서 본 M_MC=10^7 추천값보다 작게 잡음 - 속도 우선)


def marginal_one_rep(dgp, b, eps_sample):
    """training rep b 하나: 학습 -> reference -> calibration(r=1) -> 평가입력 N_eval개에서
    다섯 구간·지표·분해를 계산해 평균 (한 b의 marginal 요약값들을 long-format 행으로 반환)."""
    rng_train = substream(dgp.name, b, "train")
    rng_model = substream(dgp.name, b, "model")
    X_train, y_train = dgp.sample_xy(N_TRAIN, rng_train)
    models = {"OLS": fit_ols(X_train, y_train), "RF": fit_rf(X_train, y_train, rng_model)}

    ref_X = draw_population_reference(M_POP, substream(dgp.name, b, "reference"))
    X_cal, y_cal = dgp.sample_xy(N_CAL, substream(dgp.name, b, 1, "calibration"))
    eval_X = draw_X(N_EVAL, substream(dgp.name, b, "marginal_eval"))

    rows = []
    for alpha in ALPHAS:
        L_o, U_o = oracle_interval(eval_X, dgp, alpha)
        L_m, U_m = mc_interval(eval_X, dgp, alpha, eps_sample)
        L_r, U_r = res_true_interval(eval_X, dgp, alpha)

        for model_id, model in models.items():
            q_pop = compute_q_pop(model, ref_X, dgp, alpha)
            q_hat, state = cp_quantile(model, X_cal, y_cal, alpha)
            L_p, U_p = pop_interval(eval_X, model, q_pop)
            L_c, U_c = cp_interval(eval_X, model, q_hat)

            families = {"oracle": (L_o, U_o), "mc": (L_m, U_m), "res_true": (L_r, U_r),
                        "pop": (L_p, U_p), "cp": (L_c, U_c)}
            metric_means = {}
            for fam, (L, U) in families.items():
                cov = conditional_coverage(L, U, eval_X, dgp)
                metric_means[fam] = dict(coverage=cov.mean(), center=((L + U) / 2).mean(),
                                          half_width=((U - L) / 2).mean(), length=(U - L).mean())
            for metric in ["coverage", "center", "half_width", "length"]:
                T = {fam: metric_means[fam][metric] for fam in families}
                comps = decompose(T)
                rows.append(dict(b=b, model_id=model_id, alpha=alpha, metric=metric, cp_state=state, **T, **comps))
    return rows

In [ ]:
import os
import time

RUN_MARGINAL_FULL = True  # True로 바꾸면 B=200을 처음부터 다시 계산 (RF 1000 trees x 200회, 약 35분 소요)
MARGINAL_OUT_PATH = "../results/marginal_linear_homo_gaussian.csv"

if RUN_MARGINAL_FULL or not os.path.exists(MARGINAL_OUT_PATH):
    dgp = DGPS["linear_homo_gaussian"]
    eps_sample = draw_mc_benchmark(dgp, M_MC_ROUGH, substream("mc_benchmark", dgp.error_name))

    t_start = time.time()
    all_rows = []
    for b in range(1, 200 + 1):
        all_rows.extend(marginal_one_rep(dgp, b, eps_sample))
        if b % 20 == 0:
            print(f"b={b}/200  elapsed={time.time()-t_start:.0f}s", flush=True)

    pd.DataFrame(all_rows).to_csv(MARGINAL_OUT_PATH, index=False)
    print(f"완료. {time.time()-t_start:.0f}초. {MARGINAL_OUT_PATH}에 저장.")
else:
    print(f"RUN_MARGINAL_FULL=False이고 이미 결과 파일이 있어 재계산 생략: {MARGINAL_OUT_PATH}")


In [26]:
from IPython.display import display

marginal_perb = pd.read_csv(r"../results/marginal_linear_homo_gaussian.csv")
B_ACTUAL = marginal_perb["b"].nunique()
print(f"불러온 반복 수 B = {B_ACTUAL}")

VALUE_COLS = ["oracle", "mc", "res_true", "pop", "cp",
              "total_mc", "structural", "fitting", "calibration", "mc_approximation"]

# b들에 걸쳐 평균과 MCSE(=SD/sqrt(B))를 계산 -- "반복 간 변동"을 나타내는 게 MCSE
marginal_mean = marginal_perb.groupby(["model_id", "alpha", "metric"])[VALUE_COLS].mean()
marginal_mcse = marginal_perb.groupby(["model_id", "alpha", "metric"])[VALUE_COLS].std() / np.sqrt(B_ACTUAL)

# "평균 ± MCSE" 문자열로 합쳐서 하나의 표로 보기 좋게 만듦
marginal_summary = marginal_mean.copy()
for col in VALUE_COLS:
    marginal_summary[col] = marginal_mean[col].map("{:.4f}".format) + " ± " + marginal_mcse[col].map("{:.4f}".format)

for model_id in ["OLS", "RF"]:
    print(f"\n{'=' * 10} model = {model_id} {'=' * 10}")
    display(marginal_summary.loc[model_id])


불러온 반복 수 B = 200

========== model = OLS ==========


oracle               mc          res_true  \
alpha  metric                                                            
0.0100 center      -0.0002 ± 0.0019  0.0062 ± 0.0019  -0.0002 ± 0.0019   
       coverage     0.9900 ± 0.0000  0.9900 ± 0.0000   0.9900 ± 0.0000   
       half_width   2.5758 ± 0.0000  2.5771 ± 0.0000   2.5758 ± 0.0000   
       length       5.1517 ± 0.0000  5.1542 ± 0.0000   5.1517 ± 0.0000   
0.0500 center      -0.0002 ± 0.0019  0.0007 ± 0.0019  -0.0002 ± 0.0019   
       coverage     0.9500 ± 0.0000  0.9499 ± 0.0000   0.9500 ± 0.0000   
       half_width   1.9600 ± 0.0000  1.9595 ± 0.0000   1.9600 ± 0.0000   
       length       3.9199 ± 0.0000  3.9189 ± 0.0000   3.9199 ± 0.0000   
0.1000 center      -0.0002 ± 0.0019  0.0006 ± 0.0019  -0.0002 ± 0.0019   
       coverage     0.9000 ± 0.0000  0.8999 ± 0.0000   0.9000 ± 0.0000   
       half_width   1.6449 ± 0.0000  1.6443 ± 0.0000   1.6449 ± 0.0000   
       length       3.2897 ± 0.0000  3.2887 ± 0.0000   3.2897 ± 0.0000   

                               pop               cp          total_mc  \
alpha  metric                                                           
0.0100 center      0.0000 ± 0.0032  0.0000 ± 0.0032  -0.0062 ± 0.0025   
       coverage    0.9900 ± 0.0000  0.9899 ± 0.0002  -0.0001 ± 0.0002   
       half_width  2.5800 ± 0.0002  2.5952 ± 0.0085   0.0181 ± 0.0085   
       length      5.1600 ± 0.0005  5.1904 ± 0.0170   0.0362 ± 0.0170   
0.0500 center      0.0000 ± 0.0032  0.0000 ± 0.0032  -0.0007 ± 0.0025   
       coverage    0.9500 ± 0.0000  0.9505 ± 0.0005   0.0005 ± 0.0005   
       half_width  1.9632 ± 0.0002  1.9701 ± 0.0040   0.0107 ± 0.0040   
       length      3.9263 ± 0.0004  3.9402 ± 0.0079   0.0213 ± 0.0079   
0.1000 center      0.0000 ± 0.0032  0.0000 ± 0.0032  -0.0005 ± 0.0025   
       coverage    0.9000 ± 0.0000  0.9011 ± 0.0006   0.0012 ± 0.0006   
       half_width  1.6475 ± 0.0001  1.6543 ± 0.0030   0.0099 ± 0.0030   
       length      3.2951 ± 0.0003  3.3086 ± 0.0060   0.0199 ± 0.0060   

                         structural           fitting       calibration  \
alpha  metric                                                             
0.0100 center       0.0000 ± 0.0000   0.0003 ± 0.0025   0.0000 ± 0.0000   
       coverage     0.0000 ± 0.0000  -0.0000 ± 0.0000  -0.0001 ± 0.0002   
       half_width   0.0000 ± 0.0000   0.0042 ± 0.0002   0.0152 ± 0.0085   
       length       0.0000 ± 0.0000   0.0084 ± 0.0005   0.0304 ± 0.0170   
0.0500 center       0.0000 ± 0.0000   0.0003 ± 0.0025  -0.0000 ± 0.0000   
       coverage    -0.0000 ± 0.0000  -0.0000 ± 0.0000   0.0005 ± 0.0005   
       half_width  -0.0000 ± 0.0000   0.0032 ± 0.0002   0.0070 ± 0.0039   
       length      -0.0000 ± 0.0000   0.0064 ± 0.0004   0.0139 ± 0.0079   
0.1000 center       0.0000 ± 0.0000   0.0003 ± 0.0025   0.0000 ± 0.0000   
       coverage    -0.0000 ± 0.0000  -0.0000 ± 0.0000   0.0011 ± 0.0006   
       half_width  -0.0000 ± 0.0000   0.0027 ± 0.0001   0.0068 ± 0.0030   
       length      -0.0000 ± 0.0000   0.0054 ± 0.0003   0.0135 ± 0.0059   

                   mc_approximation  
alpha  metric                        
0.0100 center      -0.0064 ± 0.0000  
       coverage    -0.0000 ± 0.0000  
       half_width  -0.0013 ± 0.0000  
       length      -0.0026 ± 0.0000  
0.0500 center      -0.0010 ± 0.0000  
       coverage     0.0001 ± 0.0000  
       half_width   0.0005 ± 0.0000  
       length       0.0010 ± 0.0000  
0.1000 center      -0.0008 ± 0.0000  
       coverage     0.0001 ± 0.0000  
       half_width   0.0005 ± 0.0000  
       length       0.0010 ± 0.0000


========== model = RF ==========


oracle               mc          res_true  \
alpha  metric                                                            
0.0100 center      -0.0002 ± 0.0019  0.0062 ± 0.0019  -0.0002 ± 0.0019   
       coverage     0.9900 ± 0.0000  0.9900 ± 0.0000   0.9900 ± 0.0000   
       half_width   2.5758 ± 0.0000  2.5771 ± 0.0000   2.5758 ± 0.0000   
       length       5.1517 ± 0.0000  5.1542 ± 0.0000   5.1517 ± 0.0000   
0.0500 center      -0.0002 ± 0.0019  0.0007 ± 0.0019  -0.0002 ± 0.0019   
       coverage     0.9500 ± 0.0000  0.9499 ± 0.0000   0.9500 ± 0.0000   
       half_width   1.9600 ± 0.0000  1.9595 ± 0.0000   1.9600 ± 0.0000   
       length       3.9199 ± 0.0000  3.9189 ± 0.0000   3.9199 ± 0.0000   
0.1000 center      -0.0002 ± 0.0019  0.0006 ± 0.0019  -0.0002 ± 0.0019   
       coverage     0.9000 ± 0.0000  0.8999 ± 0.0000   0.9000 ± 0.0000   
       half_width   1.6449 ± 0.0000  1.6443 ± 0.0000   1.6449 ± 0.0000   
       length       3.2897 ± 0.0000  3.2887 ± 0.0000   3.2897 ± 0.0000   

                               pop               cp          total_mc  \
alpha  metric                                                           
0.0100 center      0.0014 ± 0.0034  0.0014 ± 0.0034  -0.0048 ± 0.0028   
       coverage    0.9900 ± 0.0000  0.9899 ± 0.0002  -0.0001 ± 0.0002   
       half_width  2.8390 ± 0.0013  2.8519 ± 0.0081   0.2748 ± 0.0081   
       length      5.6780 ± 0.0027  5.7038 ± 0.0162   0.5496 ± 0.0162   
0.0500 center      0.0014 ± 0.0034  0.0014 ± 0.0034   0.0006 ± 0.0028   
       coverage    0.9500 ± 0.0000  0.9498 ± 0.0005  -0.0002 ± 0.0005   
       half_width  2.1588 ± 0.0010  2.1606 ± 0.0048   0.2011 ± 0.0048   
       length      4.3176 ± 0.0020  4.3211 ± 0.0096   0.4022 ± 0.0096   
0.1000 center      0.0014 ± 0.0034  0.0014 ± 0.0034   0.0008 ± 0.0028   
       coverage    0.9000 ± 0.0001  0.9010 ± 0.0007   0.0011 ± 0.0007   
       half_width  1.8113 ± 0.0009  1.8188 ± 0.0039   0.1745 ± 0.0039   
       length      3.6225 ± 0.0017  3.6377 ± 0.0078   0.3490 ± 0.0078   

                         structural           fitting       calibration  \
alpha  metric                                                             
0.0100 center       0.0000 ± 0.0000   0.0016 ± 0.0028   0.0000 ± 0.0000   
       coverage     0.0000 ± 0.0000  -0.0000 ± 0.0000  -0.0001 ± 0.0002   
       half_width   0.0000 ± 0.0000   0.2632 ± 0.0013   0.0129 ± 0.0081   
       length       0.0000 ± 0.0000   0.5263 ± 0.0027   0.0259 ± 0.0162   
0.0500 center       0.0000 ± 0.0000   0.0016 ± 0.0028  -0.0000 ± 0.0000   
       coverage    -0.0000 ± 0.0000  -0.0000 ± 0.0000  -0.0002 ± 0.0005   
       half_width  -0.0000 ± 0.0000   0.1988 ± 0.0010   0.0017 ± 0.0046   
       length      -0.0000 ± 0.0000   0.3977 ± 0.0020   0.0035 ± 0.0092   
0.1000 center       0.0000 ± 0.0000   0.0016 ± 0.0028   0.0000 ± 0.0000   
       coverage    -0.0000 ± 0.0000  -0.0000 ± 0.0001   0.0010 ± 0.0007   
       half_width  -0.0000 ± 0.0000   0.1664 ± 0.0009   0.0076 ± 0.0037   
       length      -0.0000 ± 0.0000   0.3328 ± 0.0017   0.0151 ± 0.0074   

                   mc_approximation  
alpha  metric                        
0.0100 center      -0.0064 ± 0.0000  
       coverage    -0.0000 ± 0.0000  
       half_width  -0.0013 ± 0.0000  
       length      -0.0026 ± 0.0000  
0.0500 center      -0.0010 ± 0.0000  
       coverage     0.0001 ± 0.0000  
       half_width   0.0005 ± 0.0000  
       length       0.0010 ± 0.0000  
0.1000 center      -0.0008 ± 0.0000  
       coverage     0.0001 ± 0.0000  
       half_width   0.0005 ± 0.0000  
       length       0.0010 ± 0.0000

### 검증: marginal coverage의 structural·fitting 평균은 이론상 0

CLAUDE.md §7 "정확한 population 기준에서 marginal coverage의 structural·fitting 평균은 0". 이는 **coverage에만** 적용되고 (길이 등 다른 지표에는 적용 안 됨), 위치별로 과소·과대가 있어도 평균에서 상쇄되어 0이 될 수 있다는 뜻이지 위치별 문제가 없다는 뜻은 아님. 근사(유한 $B$, $N_{eval}$, $M_{pop}$)라서 정확히 0은 아니고 MCSE 범위 안에 있는지만 확인.

In [27]:
coverage_check = marginal_perb.query("metric == 'coverage'").groupby(["model_id", "alpha"])[
    ["structural", "fitting"]
].agg(["mean", "std"])

for model_id in ["OLS", "RF"]:
    for alpha in [0.10, 0.05, 0.01]:
        row = coverage_check.loc[(model_id, alpha)]
        structural_mean, structural_se = row[("structural", "mean")], row[("structural", "std")] / np.sqrt(B_ACTUAL)
        fitting_mean, fitting_se = row[("fitting", "mean")], row[("fitting", "std")] / np.sqrt(B_ACTUAL)
        print(
            f"{model_id:3s} alpha={alpha:<5} "
            f"structural={structural_mean:+.5f} (mcse {structural_se:.5f})   "
            f"fitting={fitting_mean:+.5f} (mcse {fitting_se:.5f})"
        )


OLS alpha=0.1   structural=-0.00000 (mcse 0.00000)   fitting=-0.00000 (mcse 0.00000)
OLS alpha=0.05  structural=-0.00000 (mcse 0.00000)   fitting=-0.00000 (mcse 0.00000)
OLS alpha=0.01  structural=+0.00000 (mcse 0.00000)   fitting=-0.00000 (mcse 0.00000)
RF  alpha=0.1   structural=-0.00000 (mcse 0.00000)   fitting=-0.00002 (mcse 0.00007)
RF  alpha=0.05  structural=-0.00000 (mcse 0.00000)   fitting=-0.00002 (mcse 0.00005)
RF  alpha=0.01  structural=+0.00000 (mcse 0.00000)   fitting=-0.00001 (mcse 0.00002)


## 12. Fixed-x 관점 ($b=1,\ldots,10$, $R_{cal}=500$)

사전 지정한 training id $b=1,\ldots,10$마다: fitted model과 population reference를 **고정**한 채, calibration만 $r=1,\ldots,500$번 새로 뽑아서 25개 고정 위치 $\{-0.9,-0.5,0,0.5,0.9\}^2$에서 다섯 구간·지표·분해를 계산. `structural`·`fitting`은 같은 $b$ 안에서 $r$과 무관하게 고정되어야 하고(계산에 이미 그렇게 구현됨), `calibration`과 `total_mc`만 $r$에 따라 변동 — 이 변동의 평균·SD를 보고한다.

각 $b$의 500회 결과는 **따로** 보고하며, $10\times500$을 독립 training 5000회처럼 합치지 않는다 (CLAUDE.md §7, §9.3).

계산은 백그라운드 스크립트로 `results/fixed_x_linear_homo_gaussian.csv`에 저장해뒀고, 여기서는 읽어서 집계·검증만 한다. 이 표의 $r=1$ 부분은 다음 §13 fixed-calibration에서 그대로 재사용.

### 실행 코드 (이미 계산된 결과가 있으면 재계산 생략)

marginal과 같은 방식: `RUN_FIXED_X_FULL = True`로 바꾸면 다시 계산(약 20분), 기본값은 저장된 결과 재사용. 여기서 저장하는 건 원본 구간값이 아니라 **분해 결과(component)만** — §9/§10처럼 원본 다섯 구간 값이 필요하면 `oracle_interval` 등을 직접 다시 호출하면 됨 (r-invariant라서 비용도 작음).

In [ ]:
import itertools

FIXED_X_COORDS = [-0.9, -0.5, 0.0, 0.5, 0.9]
FIXED_POINTS = np.array(list(itertools.product(FIXED_X_COORDS, FIXED_X_COORDS)))  # 25 x 2
FIXED_X_TRAINING_IDS = list(range(1, 11))
R_CAL = 500


def fixed_x_one_training(dgp, b, R_cal, eps_sample):
    """training b 하나 고정: 25개 fixed point에서, r=1..R_cal 500번 calibration만 바꿔가며
    다섯 구간·지표·분해(component만)를 계산. r-invariant인 oracle/mc/res_true/pop은 r 루프 밖에서 한 번만 계산."""
    rng_train = substream(dgp.name, b, "train")
    rng_model = substream(dgp.name, b, "model")
    X_train, y_train = dgp.sample_xy(N_TRAIN, rng_train)
    models = {"OLS": fit_ols(X_train, y_train), "RF": fit_rf(X_train, y_train, rng_model)}
    ref_X = draw_population_reference(M_POP, substream(dgp.name, b, "reference"))

    g_at_points = {model_id: model.predict(FIXED_POINTS) for model_id, model in models.items()}
    invariant = {}
    for alpha in ALPHAS:
        L_o, U_o = oracle_interval(FIXED_POINTS, dgp, alpha)
        L_m, U_m = mc_interval(FIXED_POINTS, dgp, alpha, eps_sample)
        L_r, U_r = res_true_interval(FIXED_POINTS, dgp, alpha)
        for model_id, model in models.items():
            q_pop = compute_q_pop(model, ref_X, dgp, alpha)
            g = g_at_points[model_id]
            invariant[(alpha, model_id)] = dict(
                oracle=(L_o, U_o), mc=(L_m, U_m), res_true=(L_r, U_r), pop=(g - q_pop, g + q_pop),
            )

    all_frames = []
    for r in range(1, R_cal + 1):
        X_cal, y_cal = dgp.sample_xy(N_CAL, substream(dgp.name, b, r, "calibration"))

        # model_id를 alpha보다 바깥에 둬서 residual 정렬을 (r, model)당 한 번만 함 (3배 중복 predict 방지)
        for model_id, model in models.items():
            residuals_sorted = np.sort(np.abs(y_cal - model.predict(X_cal)))
            g = g_at_points[model_id]

            for alpha in ALPHAS:
                m = len(residuals_sorted)
                k = math.ceil((m + 1) * (1 - alpha) - 1e-9)
                q_hat = np.inf if k > m else residuals_sorted[k - 1]
                L_c, U_c = g - q_hat, g + q_hat
                fams = dict(invariant[(alpha, model_id)])
                fams["cp"] = (L_c, U_c)

                metric_arrays = {}
                for fam, (L, U) in fams.items():
                    cov = conditional_coverage(L, U, FIXED_POINTS, dgp)
                    metric_arrays[fam] = dict(coverage=cov, center=(L + U) / 2,
                                               half_width=(U - L) / 2, length=U - L)

                for metric in ["coverage", "center", "half_width", "length"]:
                    T = {fam: metric_arrays[fam][metric] for fam in fams}  # 각 값이 (25,) 배열
                    comps = decompose(T)  # decompose는 배열에도 그대로 동작 (numpy 벡터 연산)
                    frame = pd.DataFrame({
                        "b": b, "r": r, "x1": FIXED_POINTS[:, 0], "x2": FIXED_POINTS[:, 1],
                        "model_id": model_id, "alpha": alpha, "metric": metric,
                        **comps,
                    })
                    all_frames.append(frame)
    return pd.concat(all_frames, ignore_index=True)


In [ ]:
RUN_FIXED_X_FULL = False  # True로 바꾸면 b=1..10 x R_cal=500 전체를 다시 계산 (약 20분 소요)
FIXED_X_OUT_PATH = "../results/fixed_x_linear_homo_gaussian.csv"

if RUN_FIXED_X_FULL or not os.path.exists(FIXED_X_OUT_PATH):
    dgp = DGPS["linear_homo_gaussian"]
    eps_sample = draw_mc_benchmark(dgp, M_MC_ROUGH, substream("mc_benchmark", dgp.error_name))

    t_start = time.time()
    b_frames = []
    for b in FIXED_X_TRAINING_IDS:
        t_b = time.time()
        b_frames.append(fixed_x_one_training(dgp, b, R_CAL, eps_sample))
        print(f"b={b} done in {time.time()-t_b:.0f}s  total_elapsed={time.time()-t_start:.0f}s", flush=True)

    pd.concat(b_frames, ignore_index=True).to_csv(FIXED_X_OUT_PATH, index=False)
    print(f"완료. {time.time()-t_start:.0f}초. {FIXED_X_OUT_PATH}에 저장.")
else:
    print(f"RUN_FIXED_X_FULL=False이고 이미 결과 파일이 있어 재계산 생략: {FIXED_X_OUT_PATH}")


In [28]:
fixed_x_raw = pd.read_csv(r"../results/fixed_x_linear_homo_gaussian.csv")
print(f"불러온 행 수 = {len(fixed_x_raw):,}  (b={fixed_x_raw['b'].nunique()}, r={fixed_x_raw['r'].nunique()}, "
      f"위치={fixed_x_raw[['x1','x2']].drop_duplicates().shape[0]})")

# 불변식 재확인: structural/fitting은 r과 무관 -> 같은 (b,x,model,alpha,metric)에서 r에 대한 SD가 0이어야 함
invariance_check = fixed_x_raw.groupby(["b", "x1", "x2", "model_id", "alpha", "metric"])[
    ["structural", "fitting"]
].std()
print("structural/fitting의 r에 대한 SD 최댓값 (0이어야 함):", invariance_check.max().max())

center_calibration = fixed_x_raw.query("metric == 'center'")["calibration"]
print("center의 calibration 최대 절댓값 (0이어야 함):", center_calibration.abs().max())

max_identity_residual = fixed_x_raw["identity_residual"].abs().max()
print("max |identity_residual| (0이어야 함):", max_identity_residual)


불러온 행 수 = 3,000,000  (b=10, r=500, 위치=25)
structural/fitting의 r에 대한 SD 최댓값 (0이어야 함): 0.0
center의 calibration 최대 절댓값 (0이어야 함): 8.881784197001252e-16
max |identity_residual| (0이어야 함): 2.220446049250313e-16


### 대표 사례: $b=1$, $x=(0,0)$에서 500회 calibration 반복의 평균·SD

10개 $b$ × 25개 위치 전부를 한 번에 보여주면 너무 많으므로, 대표로 $b=1$, 위치 $(0,0)$만 골라 $R_{cal}=500$번의 calibration 반복에서 각 지표·분해항의 평균과 SD를 봄. 다른 $b$·위치는 `fixed_x_raw`에 전부 저장되어 있어 같은 방식으로 꺼내 볼 수 있음.

In [29]:
# fixed_x_raw에는 분해 결과(component)만 저장되어 있음 (원본 oracle/mc/res_true/pop/cp 값은 없음)
COMPONENT_COLS = ["total_mc", "structural", "fitting", "calibration", "mc_approximation"]

example = fixed_x_raw.query("b == 1 and x1 == 0.0 and x2 == 0.0")

example_mean = example.groupby(["model_id", "alpha", "metric"])[COMPONENT_COLS].mean()
example_std = example.groupby(["model_id", "alpha", "metric"])[COMPONENT_COLS].std()

example_summary = example_mean.copy()
for col in COMPONENT_COLS:
    example_summary[col] = example_mean[col].map("{:.4f}".format) + " ± " + example_std[col].map("{:.4f}".format)

for model_id in ["OLS", "RF"]:
    print(f"\n{'=' * 10} model = {model_id}  (b=1, x=(0,0), over r=1..500) {'=' * 10}")
    display(example_summary.loc[model_id])



========== model = OLS  (b=1, x=(0,0), over r=1..500) ==========


total_mc        structural           fitting  \
alpha  metric                                                             
0.0100 center      -0.0221 ± 0.0000   0.0000 ± 0.0000  -0.0157 ± 0.0000   
       coverage    -0.0002 ± 0.0033   0.0000 ± 0.0000   0.0001 ± 0.0000   
       half_width   0.0084 ± 0.1135   0.0000 ± 0.0000   0.0031 ± 0.0000   
       length       0.0168 ± 0.2270   0.0000 ± 0.0000   0.0061 ± 0.0000   
0.0500 center      -0.0167 ± 0.0000   0.0000 ± 0.0000  -0.0157 ± 0.0000   
       coverage    -0.0004 ± 0.0068  -0.0000 ± 0.0000   0.0002 ± 0.0000   
       half_width   0.0001 ± 0.0578  -0.0000 ± 0.0000   0.0023 ± 0.0000   
       length       0.0002 ± 0.1155  -0.0000 ± 0.0000   0.0047 ± 0.0000   
0.1000 center      -0.0165 ± 0.0000   0.0000 ± 0.0000  -0.0157 ± 0.0000   
       coverage     0.0002 ± 0.0089  -0.0000 ± 0.0000   0.0004 ± 0.0000   
       half_width   0.0029 ± 0.0435  -0.0000 ± 0.0000   0.0020 ± 0.0000   
       length       0.0058 ± 0.0870  -0.0000 ± 0.0000   0.0039 ± 0.0000   

                        calibration  mc_approximation  
alpha  metric                                          
0.0100 center       0.0000 ± 0.0000  -0.0064 ± 0.0000  
       coverage    -0.0003 ± 0.0033  -0.0000 ± 0.0000  
       half_width   0.0066 ± 0.1135  -0.0013 ± 0.0000  
       length       0.0132 ± 0.2270  -0.0026 ± 0.0000  
0.0500 center       0.0000 ± 0.0000  -0.0010 ± 0.0000  
       coverage    -0.0007 ± 0.0068   0.0001 ± 0.0000  
       half_width  -0.0027 ± 0.0578   0.0005 ± 0.0000  
       length      -0.0055 ± 0.1155   0.0010 ± 0.0000  
0.1000 center       0.0000 ± 0.0000  -0.0008 ± 0.0000  
       coverage    -0.0002 ± 0.0089   0.0001 ± 0.0000  
       half_width   0.0004 ± 0.0435   0.0005 ± 0.0000  
       length       0.0008 ± 0.0870   0.0010 ± 0.0000


========== model = RF  (b=1, x=(0,0), over r=1..500) ==========


total_mc        structural           fitting  \
alpha  metric                                                             
0.0100 center      -0.3164 ± 0.0000   0.0000 ± 0.0000  -0.3100 ± 0.0000   
       coverage     0.0027 ± 0.0024   0.0000 ± 0.0000   0.0029 ± 0.0000   
       half_width   0.2455 ± 0.1191   0.0000 ± 0.0000   0.2361 ± 0.0000   
       length       0.4911 ± 0.2381   0.0000 ± 0.0000   0.4722 ± 0.0000   
0.0500 center      -0.3109 ± 0.0000   0.0000 ± 0.0000  -0.3100 ± 0.0000   
       coverage     0.0090 ± 0.0061  -0.0000 ± 0.0000   0.0091 ± 0.0000   
       half_width   0.1812 ± 0.0630  -0.0000 ± 0.0000   0.1783 ± 0.0000   
       length       0.3623 ± 0.1260  -0.0000 ± 0.0000   0.3567 ± 0.0000   
0.1000 center      -0.3108 ± 0.0000   0.0000 ± 0.0000  -0.3100 ± 0.0000   
       coverage     0.0136 ± 0.0090  -0.0000 ± 0.0000   0.0134 ± 0.0000   
       half_width   0.1521 ± 0.0512  -0.0000 ± 0.0000   0.1492 ± 0.0000   
       length       0.3041 ± 0.1024  -0.0000 ± 0.0000   0.2984 ± 0.0000   

                        calibration  mc_approximation  
alpha  metric                                          
0.0100 center       0.0000 ± 0.0000  -0.0064 ± 0.0000  
       coverage    -0.0001 ± 0.0024  -0.0000 ± 0.0000  
       half_width   0.0107 ± 0.1191  -0.0013 ± 0.0000  
       length       0.0214 ± 0.2381  -0.0026 ± 0.0000  
0.0500 center       0.0000 ± 0.0000  -0.0010 ± 0.0000  
       coverage    -0.0001 ± 0.0061   0.0001 ± 0.0000  
       half_width   0.0023 ± 0.0630   0.0005 ± 0.0000  
       length       0.0046 ± 0.1260   0.0010 ± 0.0000  
0.1000 center      -0.0000 ± 0.0000  -0.0008 ± 0.0000  
       coverage     0.0000 ± 0.0090   0.0001 ± 0.0000  
       half_width   0.0023 ± 0.0512   0.0005 ± 0.0000  
       length       0.0047 ± 0.1024   0.0010 ± 0.0000

## 13. Fixed-calibration 관점

fitted model과 calibration을 **고정**하고 위치 $x$를 바꿔가며 보는 관점. §12의 fixed-x 결과에서 **$r=1$만 뽑으면** 그대로 이 관점의 데이터가 된다 (같은 fitted model+calibration을 재사용하는 것은 동일하므로 재계산 불필요).

- **individual_fixed_realization**: 하나의 특정 $(b,r)=(1,1)$ 조합 — 25개 위치별 값을 그대로 봄 (평균 내지 않음)
- **training_calibration_averaged**: $b=1,\ldots,10$ 각각의 $r=1$ calibration에서 나온 값을, 위치별로 $b$에 대해 평균·SD

CLAUDE.md 스펙은 이 "반복 평균"에 기본 $B=200$ 전체를 쓰라고 하지만, 이번 rough pass에서는 §12(fixed-x)와 계산을 공유하기 위해 **$b=1,\ldots,10$만 사용** — 정직하게 축소된 범위로 표시. 전체 $B=200$으로 확장하려면 §11(marginal)에서 이미 계산한 200개 training의 $r=1$ 결과를 25개 고정점에서도 뽑도록 나중에 확장하면 됨.

In [30]:
# fixed_x_raw에는 분해 결과(component)만 있음 -- 원본 다섯 구간 값은 §9/§10처럼 별도로 계산해야 봄
COMPONENT_COLS = ["total_mc", "structural", "fitting", "calibration", "mc_approximation"]

# individual_fixed_realization: (b, r) = (1, 1) 하나만, 25개 위치별 값 그대로
individual_fixed_realization = fixed_x_raw.query("b == 1 and r == 1").drop(columns=["b", "r"])
print("=== individual_fixed_realization: (b=1, r=1), coverage, alpha=0.05, RF ===")
display(
    individual_fixed_realization.query("metric == 'coverage' and alpha == 0.05 and model_id == 'RF'")
    .set_index(["x1", "x2"])[COMPONENT_COLS]
)

# training_calibration_averaged: r=1만 골라서 b=1..10에 대해 위치별 평균·SD
r1_rows = fixed_x_raw.query("r == 1")
n_b = r1_rows["b"].nunique()
tc_mean = r1_rows.groupby(["x1", "x2", "model_id", "alpha", "metric"])[COMPONENT_COLS].mean()
tc_std = r1_rows.groupby(["x1", "x2", "model_id", "alpha", "metric"])[COMPONENT_COLS].std()

tc_summary = tc_mean.copy()
for col in COMPONENT_COLS:
    tc_summary[col] = tc_mean[col].map("{:.4f}".format) + " ± " + tc_std[col].map("{:.4f}".format)

print(f"\n=== training_calibration_averaged (b=1..{n_b}의 r=1), coverage, alpha=0.05, RF ===")
display(
    tc_summary.xs(("RF", 0.05, "coverage"), level=("model_id", "alpha", "metric"))
)


=== individual_fixed_realization: (b=1, r=1), coverage, alpha=0.05, RF ===


total_mc  structural  fitting  calibration  mc_approximation
x1      x2                                                                   
-0.9000 -0.9000    0.0142     -0.0000   0.0156      -0.0015            0.0001
        -0.5000    0.0151     -0.0000   0.0165      -0.0015            0.0001
        0.0000    -0.1492     -0.0000  -0.1442      -0.0050            0.0001
        0.5000     0.0156     -0.0000   0.0171      -0.0015            0.0001
        0.9000    -0.0186     -0.0000  -0.0162      -0.0024            0.0001
-0.5000 -0.9000   -0.0793     -0.0000  -0.0755      -0.0038            0.0001
        -0.5000    0.0147     -0.0000   0.0162      -0.0015            0.0001
        0.0000     0.0032     -0.0000   0.0050      -0.0019            0.0001
        0.5000     0.0089     -0.0000   0.0105      -0.0017            0.0001
        0.9000    -0.1006     -0.0000  -0.0964      -0.0042            0.0001
0.0000  -0.9000    0.0018     -0.0000   0.0036      -0.0019            0.0001
        -0.5000    0.0149     -0.0000   0.0164      -0.0015            0.0001
        0.0000     0.0074     -0.0000   0.0091      -0.0017            0.0001
        0.5000     0.0046     -0.0000   0.0064      -0.0018            0.0001
        0.9000    -0.0096     -0.0000  -0.0075      -0.0022            0.0001
0.5000  -0.9000    0.0064     -0.0000   0.0081      -0.0018            0.0001
        -0.5000   -0.0087     -0.0000  -0.0066      -0.0022            0.0001
        0.0000    -0.0047     -0.0000  -0.0027      -0.0021            0.0001
        0.5000     0.0159     -0.0000   0.0173      -0.0015            0.0001
        0.9000     0.0156     -0.0000   0.0171      -0.0015            0.0001
0.9000  -0.9000   -0.0445     -0.0000  -0.0415      -0.0031            0.0001
        -0.5000    0.0152     -0.0000   0.0167      -0.0015            0.0001
        0.0000     0.0140     -0.0000   0.0155      -0.0016            0.0001
        0.5000     0.0124     -0.0000   0.0139      -0.0016            0.0001
        0.9000     0.0138     -0.0000   0.0153      -0.0016            0.0001


=== training_calibration_averaged (b=1..10의 r=1), coverage, alpha=0.05, RF ===


total_mc        structural           fitting  \
x1      x2                                                              
-0.9000 -0.9000  -0.0129 ± 0.0324  -0.0000 ± 0.0000  -0.0096 ± 0.0338   
        -0.5000   0.0036 ± 0.0191  -0.0000 ± 0.0000   0.0070 ± 0.0163   
        0.0000   -0.0155 ± 0.0491  -0.0000 ± 0.0000  -0.0121 ± 0.0489   
        0.5000    0.0066 ± 0.0116  -0.0000 ± 0.0000   0.0099 ± 0.0091   
        0.9000   -0.0142 ± 0.0303  -0.0000 ± 0.0000  -0.0097 ± 0.0264   
-0.5000 -0.9000   0.0000 ± 0.0288  -0.0000 ± 0.0000   0.0029 ± 0.0290   
        -0.5000  -0.0058 ± 0.0451  -0.0000 ± 0.0000  -0.0020 ± 0.0415   
        0.0000   -0.0014 ± 0.0250  -0.0000 ± 0.0000   0.0028 ± 0.0200   
        0.5000   -0.0041 ± 0.0365  -0.0000 ± 0.0000  -0.0000 ± 0.0315   
        0.9000   -0.0186 ± 0.0335  -0.0000 ± 0.0000  -0.0146 ± 0.0333   
0.0000  -0.9000  -0.0037 ± 0.0175  -0.0000 ± 0.0000  -0.0001 ± 0.0169   
        -0.5000   0.0063 ± 0.0156  -0.0000 ± 0.0000   0.0100 ± 0.0106   
        0.0000   -0.0027 ± 0.0186  -0.0000 ± 0.0000   0.0015 ± 0.0139   
        0.5000   -0.0029 ± 0.0132  -0.0000 ± 0.0000   0.0006 ± 0.0123   
        0.9000   -0.0003 ± 0.0180  -0.0000 ± 0.0000   0.0023 ± 0.0222   
0.5000  -0.9000   0.0036 ± 0.0176  -0.0000 ± 0.0000   0.0069 ± 0.0163   
        -0.5000  -0.0164 ± 0.0424  -0.0000 ± 0.0000  -0.0116 ± 0.0390   
        0.0000   -0.0004 ± 0.0300  -0.0000 ± 0.0000   0.0034 ± 0.0258   
        0.5000    0.0019 ± 0.0161  -0.0000 ± 0.0000   0.0051 ± 0.0151   
        0.9000   -0.0040 ± 0.0217  -0.0000 ± 0.0000  -0.0020 ± 0.0277   
0.9000  -0.9000  -0.0020 ± 0.0316  -0.0000 ± 0.0000   0.0017 ± 0.0297   
        -0.5000  -0.0118 ± 0.0264  -0.0000 ± 0.0000  -0.0077 ± 0.0247   
        0.0000   -0.0037 ± 0.0258  -0.0000 ± 0.0000  -0.0009 ± 0.0292   
        0.5000    0.0081 ± 0.0106  -0.0000 ± 0.0000   0.0113 ± 0.0079   
        0.9000   -0.0171 ± 0.0645  -0.0000 ± 0.0000  -0.0157 ± 0.0720   

                      calibration mc_approximation  
x1      x2                                          
-0.9000 -0.9000  -0.0034 ± 0.0077  0.0001 ± 0.0000  
        -0.5000  -0.0035 ± 0.0062  0.0001 ± 0.0000  
        0.0000   -0.0035 ± 0.0061  0.0001 ± 0.0000  
        0.5000   -0.0033 ± 0.0056  0.0001 ± 0.0000  
        0.9000   -0.0045 ± 0.0076  0.0001 ± 0.0000  
-0.5000 -0.9000  -0.0030 ± 0.0052  0.0001 ± 0.0000  
        -0.5000  -0.0038 ± 0.0068  0.0001 ± 0.0000  
        0.0000   -0.0043 ± 0.0063  0.0001 ± 0.0000  
        0.5000   -0.0041 ± 0.0069  0.0001 ± 0.0000  
        0.9000   -0.0040 ± 0.0061  0.0001 ± 0.0000  
0.0000  -0.9000  -0.0036 ± 0.0064  0.0001 ± 0.0000  
        -0.5000  -0.0037 ± 0.0059  0.0001 ± 0.0000  
        0.0000   -0.0042 ± 0.0062  0.0001 ± 0.0000  
        0.5000   -0.0036 ± 0.0066  0.0001 ± 0.0000  
        0.9000   -0.0026 ± 0.0061  0.0001 ± 0.0000  
0.5000  -0.9000  -0.0033 ± 0.0048  0.0001 ± 0.0000  
        -0.5000  -0.0049 ± 0.0063  0.0001 ± 0.0000  
        0.0000   -0.0038 ± 0.0068  0.0001 ± 0.0000  
        0.5000   -0.0033 ± 0.0058  0.0001 ± 0.0000  
        0.9000   -0.0020 ± 0.0067  0.0001 ± 0.0000  
0.9000  -0.9000  -0.0037 ± 0.0051  0.0001 ± 0.0000  
        -0.5000  -0.0042 ± 0.0067  0.0001 ± 0.0000  
        0.0000   -0.0028 ± 0.0064  0.0001 ± 0.0000  
        0.5000   -0.0033 ± 0.0052  0.0001 ± 0.0000  
        0.9000   -0.0014 ± 0.0087  0.0001 ± 0.0000

## 현재 구현 범위

구현되고 검증됨: RNG 독립 스트림, 오차분포 3종, DGP, OLS/RF 학습, 5개 구간 전부(oracle·mc·res_true·pop·cp), M_MC/M_pop 탐색, CP 순위 검증, 다섯 구간 비교표, structural/fitting/calibration/mc_approximation 4항 분해, **marginal(B=200)·fixed-x(b=1..10, R_cal=500)·fixed-calibration 세 관점 (linear_homo_gaussian 1개 DGP)**.

이번 rough pass의 알려진 축소/잠정 사항 (명시적으로 남겨둠, 조용히 숨기지 않음):
- DGP는 `linear_homo_gaussian` 1개만 (전체 12개 DGP 중). 나머지 11개는 같은 코드로 반복하면 됨
- $M_{MC}=2{,}000{,}000$, $M_{pop}=20{,}000$, $N_{eval}=2000$은 pilot 정식 검증 전 잠정값
- fixed-calibration의 "반복 평균"은 $b=1,\ldots,10$만 사용 (스펙은 $B=200$ 전체)
- 결과는 CSV로 저장했지만 seed manifest·버전 기록 등 재현성 메타데이터는 아직 없음
- Heatmap, sample-size sensitivity는 범위 밖 (의도적으로 후순위)
